# Thử nghiệm Dijkstra, Yen và BOA* trên mạng đường Thạnh Mỹ Tây

Notebook tự chứa dữ liệu và mã chạy cho **10 cặp OD** được chọn từ bộ gốc: OD01, OD02, OD03, OD04, OD06, OD07, OD13, OD14, OD17, OD21. Các cặp được đánh lại mã OD01–OD10, đồng thời giữ mã gốc để đối chiếu. Không cần `thu_nghiem.py` hoặc các file mã tham chiếu để chạy.

**Cách chạy:** mở trong Colab hoặc Jupyter với Python 3.10 trở lên, có pandas, rồi chọn **Run all**. Mặc định chạy đủ 10 OD; có thể đặt `OD_CAN_CHAY = ["OD01"]` ở mục 2 để chạy một cặp. Dữ liệu mạng gồm 509 cạnh, 458 nút; ngưỡng rẽ lớn hơn 35° và tốc độ giờ cao điểm là giả định mô phỏng.

Kết quả được tính lại và hiển thị ngay trong notebook: bảng thời gian đối chiếu, ba bảng thuật toán riêng và bảng đường chính. Ô cuối xuất CSV từ cùng kết quả trong bộ nhớ.


## 1. Nạp dữ liệu và tạo đồ thị

In [22]:
import base64
import csv
import hashlib
import json
import zlib
from pathlib import Path

# Bản chụp dữ liệu đã kiểm tra. Mốc và tốc độ được xem ở các ô bên dưới.
snapshot = (
    'c-ri}ZI2|$kskP068&j)*PG*RW^V3I+?QSQjyy<9EyOt<0b<Z-cK3EQX1aRRU0vdUByfv`us8?@i4X*V5Cn0SAb8Y@Aom40Xa>T4Xkz|H-!D1y'
    'c*}?|%gm^XsLBjU<Z`<!D%;ci33GFMw;w%t_VoF)HxItH`Tmcpzdm~LRDO8iA3WSV_-BtlYkousF~sc3)3?<FX7VKeD^IS=lMoS!62?#-T|R#K'
    'tUUeJ`!BwHxk11A-!7kTzWe*XxZM21ix=<y<ID1yix<x?<SWX@UO#*D;_0)?s~3+yDZhFB>haS@SD$^_TjI(4FMj#yruw=UZyr7V<dbLBKm73Z'
    '+h^rj_2AK)tH)Pw-|YU;)y2ENzkFVP|LXEt^HrCx%FmxXyL^6iQGVvQ4G(_ua5LCGM!o+8l)L?Vm%R(J;k-k@L*~T9$>lx-zyDWX{qFsj|6cb0'
    'jWF8w)w9>K<=3C7J@5X_lZ$u1xx5%}`+s@-$s2RuyMH)k?||FHdh_Yn=7$@tHV+ORWA+hNvUg8z8<XCAHa1Uw$ob%%-))@$L-N6+#rX{9%leeI'
    'UcY{FM-Uk9_qqjz{fP0`fdFU#Gz%o>a_+MLMwU~sJNkqUB65tekHF!5zj^lL)#cNBJOZDIpdX$AaNCd-0KpvrV6@$i1;C|{)&u};S&M++kH7!j'
    'R-4ODBqTD{r(`X=N3g_yfFs)<Znk~-C|5$FHEkc#mc8CSPB{JDUhC73z&Yj=0<UWS^7{4m>DbvHZnh`>$TP#LPCsv1>wY0jl78K8cL*e4&Z{~C'
    'h}&o!0iJ@uZnvHMccL2PRdGPtM!o$wLH>7J?Y({x3vpG~U)qu(0n((m-)(kq0lHP`Kb$QIm?z$UZm9e)iBMeC?qM4=_p_V8{oP*sFJG*}Rh|8?'
    'bz1vjKke-|n;q;wtxNX7Hj>`|MD4HHY`uT)otT8=SGE6u+k|@m$JD>!W_Q|egms;M@3vlF{Y+D|-*&H^?DOnvt7$d653tSJ@?}f+^PG-sU;c}v'
    'T_18${?YmRK0M@Y(AOXHaR_!B?)X0e4PAfhfPlAXa6z2F1>J^Q?{^>{LS7vP!JAx=jzBQn@DKyHIt=30>8l_b!oX*u4udSarR-7~Tm?jl{Xi4?'
    '`ui__^>*`}cRzoz`R2R7y&OBjU%$Ql_$~zDllNc#^6Ka<;Nupx8jy`6K-rI^fbT~D0%CMQUccc6=M*--|5q>GfBBb}n?II!f4TY7i}zpt>Wn0T'
    '^INZBkjEAd0K0S0?mG62!5q;S8_58I<9P!S!ZvA_xc7!t5M|>#qtMHbB_f8gFAx#{C+{^7A6K+&t^|S`Z+`S|m%bB3FV=U0Hv&T``+B!;cM-6Z'
    'W8W|4<jgqEFffb1{rv4`@4x)#mz!_Cc)a=W^}FAkS!hfwBh-eWHv&UL1V-DfAc&9yAi1<A07N$?;}FnT7+^64@=h3Fxd93KUa|VgFf7#E2VZ?Y'
    'X8liJzj%34k1%b?V0{8(-wZU`Hzq&Y_vp3P`>r&A4=>(*etA&~u=}_Rr&)w?8?-;zt8f^GD7)PSL{lOMnIpR%+b7O0SnvJ1(DaL|O?L>s_5RD>'
    'zTL<#e^GDkj2J==_?Fm!WUL+pm*nE_`v_fPu^Z)<80tnY4Q1i+%kZ>35#81q`%g#qKiKPT_r8%KYuUYzyk$cYj=e+yL-4+rA0&q%l=Pt8iKw1X'
    'bBU}-A{ah0KXwvT+sB06_Ul^@vs39&!w~8B-u?aNyD#2<@&6WaAaDOKHy{4~*Y_y}rq~E5N5Hrh#;zg-!f?w?iWIsSy%<P&bufV6S}2T+#1K+)'
    'fMLW6%}v;I7l%OkW}@KamqE6YMf})dq!1YHcc2x9bs^x>)+YmjIFSd!I9(1StqcK3+pOKjK8_`#ARyKt?~=wuQx1@NE;Ek^9;sy0tqcR7x5>hQ'
    'j$km_ah*Kqibq+@#+4lf1}l8v$38QI5BmG<>&l5&h5%aGlRmOSAbRY1PFrCBLx&N_{Ta2NgfdKU7`=J+`ufF_XZP(a66?TZg0VczIms)JhIKHw'
    '6{D9YZe?r$Y^en?9f4rD-C7jL$q*5d^UCxBPFuDh$VVU;?zg!XLC!%&=hg*5jN5FONonLcU>B2^)H^dNvP*Is;3$(MMJtzF^oLE+l&_C!%6H^H'
    'Hne2rIb*jaEfT<w?gj5bq<(NcnE;VF%SAu~MY-gek`+;KXB<unTZje*qXQwgLJ&2P!q|m8(30eC5M%;`p#$~ZFE&?S{ciKEcRv>tzW4a-JVzL|'
    'sBcFAXJ8QGz;sSgo={Tp6oPdL1tKH648R{lV7T7{v9%RIkgZq>035?$xZz=$&B{!s09z&l13ZSoaKi^ei!0y*-nw95KvT~GXmQ|Sb?y_i<UyE<'
    'fgf269&-Ju6=4vpoI3!XhC#dG#(&_GXc28BW<~n4-&*n@9>rj|;m(P|ML}5GX+XD)1}EU2+cBxrz`^h&8TE%8&phKIlM!J8grh(VcYWZFTnP|t'
    ')M5bI$0Fk)&*GjNqGizixhKu3c=XAO`wKeh!sM-?I0sWiCI>x!h*?Y}QL)GR0;FL++8QFy=y~h4Ne$5y5NTAPO)2zZZw)_qGQ@+AX%~ra0+C<+'
    '?yJvh0?*@MB-&=<#)uY6#3l>lZQp<fR5taHBbrie$D3)PxD)vsMU@T~PsFJ}_?@>6I0GbfR=VQZ13`Y9_3bH_!3c=#nYr;RcQIKJjQ!iQeKd^B'
    'pFAWxRj2kw_YR6vaUeH787{^3;lNhH9#T3ELx0cv9gyol0bmQ(smRA*80`64W%8{H1-UqlwkBzOZx-p7n-Ye%UAC^{+$(DZ4%vHx6j$bsPuoNW'
    '24G{5bVh&D7^LYb{U^>;@y^sCo7^0=|K2?b4ia%lxKkJae2fvv(HfL#%?jSC^CG8iN(f?G_i}mLhDM;0HoUWfGl@?Vr=1X#VXzXo$YmRYaXcc6'
    'JvkABbjGpjXytU1PZkL1a9Uo!QQ1J7W9jWDt#b)m^ac$17!0F5_nCnJRt5uagH{xX&;+4h;ZXJ~NH`3mTsF+B%jB|L@<jG|puV3W$3Wnlte$~|'
    'jj+l~qSQTD{E&}gG1&KYUxgTl>ed1USaDk#*Fb0}hWox&S<tIPg5-c07Y0WY306$u(D!2%EMg;ukXKX+M{(<;fj~Nnz;L_Q{oICCArOM$lcu<b'
    '1$K5s!XR2xaDmpn|DrB9LtRL=IEdaz@f+mxFhpodL|2sJhg(Pn2>uud!wp~8oKaj43Bua47>q8tWEj*V1cuukAi!>Q$^XeZ3OE&mL3v;=2Y5vg'
    '#BDT`gG9$5&^gZ?D7-$k*|-u?n9Kz|t=Z-1QcOOsEUhbsEm<J&(OY4-<GvB;cx8tnTdF~d#~>JPx6uw0IxioMYfIutrgJT&qkJ&j@jyGEmE|;f'
    'E4`0Wo~j&>a;`O+<cczy%rsBRZsK8RHheu)99MQTklTXMP1)NZq;Ld;>=P#M4bNIYI$a{q1a~u8by30lV>pa9JrD}PuMh`sl|lp$$AB1adMF(8'
    '`fw1H$>_2Swz80Ed@dOC5nZ8y!JTuZ99lPJKvQj`URGCWn}F00O*f9V?EEU-YUnL)Kt2Y<aN9$F!a75fW}m5m@fZ@rZ4Y^Z*NH^j0vJ%F$xOk='
    '!ZB+>!fG-RFxx=%qI2Lw^npXqtZ4O$1Xe>nnpjJCnviw$Rt85P)6+m64ZkBB+LY0mGEh46)`ZFG$QLL~fTF+c>&`J<;#IgpB!a166CVR&xaol+'
    'k!XcDWVu3(!@)KX#0R@_b#Z-fw5tpd&sV4N*~;h$c7SdKLx0PI@C@dap^(@L*z(v11B~ntd%1310yOHJXafWyf9@?@o!QVn;~0q)Sb%Wj44>Hd'
    '#6&D=1)|ZC1G7KdR4)Q6F}#`6au9>ThOe8Ju8A3Zu#C=g27$NU!XO;OV6@>OBfz>ah|$s-I428)oDPZw?=z>;^)eVmBb+zADUWa_^%2e*?Zr7C'
    '+x=*(L#gOjvU^BY)6VFev3qj8(y3uh%&=a_P?Rv7d6|sIEd~P!e+-D>u7_N}D+D51eVU?!$qZqfhO|N)>|PAfja7sUnLJYuh|H9cLqGYoeFXb^'
    '{Qmx@FE&4T|K-p6Elx*B{MoBlSLLt!Jr3<6cT{g$%F(ld3E7zVNtobGy)SRBo_+e}bRdA47Tq+s*C4<o5NKr}c(S@P#Br@e3j%5oK!Xt0#tCi<'
    'mWYTp2s8-<UK<ExWw}H*ZhP(6MQr;0)ec!iPM-RGI;xJ3AwY&Tb-o6+#^S_r%}k3x?p>Kpz1Z%`L?`;BV~_4Y3E2kDYA#F}h{6U7s6ik#0DVhx'
    'P&9bmD=W(ZZUVrmtLUnkFa`o+7LeDAMI}<*ze!b1L0Da4hY^^G;EqACzk04ICR<VVFIY{VBX}Ew(36D$TuF7?9fbA~)0$)<r7auwB*0Vzn2hwE'
    'ysAM6V9rX9<1TJB2L0M{9l-ZUqB&%sVFZ6AHTH3v43m1?$xY)h+V!BhrC(vQ0a2C>FJm0Hb8CQLI#@r~6;H+0$ws3ad>rTYXb^@QuI~p7j-2DQ'
    'CBX*MUH~;ACwb%;xz}_8J&<rhPf=_iS&hKqd>h^pZ^3XOK$?s})EK07VG!fi(v-%^BD3hn+zUqP*$}02E)j_l6>m~1cG#tM>oTzkv4-rez+1%e'
    '-Ix-J3CY90D?Twe)dk*a`HbYxI%ma~i2%mkh@-oa9AV&QhEz{S@V%+RRZ}}|%&OJc%(#aUf(Zb-;Z%qa9Yo!_Ze`{|bXK#rI8JaZ7_jqsG%~Cm'
    '5Vq<eF_3LGh+_zn2>Q%JU73>e3seJ~_#~8E1mi{=XEYWFSWVY1{v;kW<O-380DSoFuP*DxjnB_+cBZptA;fV90loFE=XWo2&pQ_T5#q3F`h0gE'
    'oKY1PgFm+S(MAt6c|PmC8@1xij(VF1k#fnqgAMPy=C)Scne&;aBH8hGc1M92?s}&UidqDgQE>K1@HQGO!TvZ)40b(iZXDLg5@DOQ7cq^U$rQmw'
    '=3YjMF(s!}lFfAHeC|oSxXoHbCZ!xi2%-@7)0v$pgR%6Y^QvcOG>{;K)0qp?&W4eklEy=(DW;fH?l-)am!E@J3p=x^d*t`$BJ3R%bK*dgH`<@4'
    '@yO{Gg?@Ed@+k|Gj3SL9A%<ycYeB&n4w1$k$yykKW`@336@A7cB;q;YFwv_@TO<PrvVq_O=2OQGusRqJ&3R&O>{1C3AV%tq(Q3yh^-`HVal$l8'
    'WnM!Mcwlp|oEw)z)hMK1ZhP=$LcMQBa$XM}8QO!-?r68d)A*FY)T!IP)srC5xtv66f<VrK(X3JIB|o0J3vwkg4n;yB>eba(l|W)jCrfkUGgl!?'
    'S7Ww-fMY-mx7}FQ!U~kjM41&rk@p~h$B-Cp`#{p-N{|R!XC4kZhQx5!b?Ge0#Sxt)9?l*IvZk=QXv09mX(4hoGU{1-R>nJ}^Nm1dv;DnX18NE>'
    '`7(EMI1Ql$95M{1LR60|%jy^ir%LyCX6|N5<gyV+8UfY_;5wpk5v_7a@5jmQ)#iy@8dN)!{++{V%=II~=>$L@EER}_<u#^kC8!7tG#Q3C<N*;H'
    'z^j8HSm`O=k2BgE7zVX3^`6mb7*-OHj$$}iJm75}0KIuiYV%}lUQF%#7lji5uxCw{q01V8QUi$t0Eon1qyYfcBcuKeGMujBXG$q(uo9HKABSTF'
    '2;qZfvSO<kU;-L6sKY=$1gmENMBgg}Jc)`h0gaTm!9*k5Xbjk*+I!Y;SY?rr)&NJgrlfcZN=oDun3)fC;OQP&i7TGZesDWFt6o^Z2sRk;P?O{C'
    'k(I-h-B~wNFnauYVrQd}6$-#AyVY4e1e`l9OH6V&IIDr3py5q4qC3>vvU_BxQo*OQ?qDZ}!&x}kXCu;-PJRK_p-Ta7v!P2dDYGKy<ihYslrUw?'
    'D!ngeE0*ox16*=M?ea>syz(t43yf%jG3@qQ$85a{<681T*ywCqOCVd#6$FGd6$CeGKNJIzRabw`DEPbu^IYU(D2%o|<b{=?kZj=)oDBjTVukKe'
    'TG@GU+h8~!oBMDK0=sZv?qut9vU>KB<MZy>$6K<PA&!ks?_2`c3tDzfcEai#Fcr(2ED+EJArHiYBZn|lni7=aX|jqT^7#r!m#vX0o|6p&pl&&m'
    '6$;i=dm{fqz=eQVUSl9@fRMmIHVoky3?+PvZVwFMY+>L%D#LhUM1upmkyAGGTXMCJf`vmo-$N16OlRQn%!n3*UO{;knDc`ipQ1xV_Q|>|Q#kja'
    'C7M8_V?d0yJ!A*JMkLU5IPiFMLJLJ!(TZRoVTJT#IA6C*1XD_8Z-PL5RT||-3aLL$rR5PTbjdEBcdvZ6WwfAx<4_E?J&58lt`7;hGO{HkvaLZ-'
    'Ka0MkcNFK7Gnm4vC%`j@!Wu;BRZ~v`iQ%q?JQ3ChM9L;lM88K+49lv5^B4zZRe;It>q+j6>x{|dhUo3fGmwsNdmZLC1Oy_VFA61LIvRpfwPk5x'
    '2*bq4kjSU0VgB=l0h7_8$g9pQ1_*;4Uk`A}D+7T<dq@@vwtNinAPViHU}t>z(`0;13SPObBMXOM<6vaI<Y@^&9?mlXEU;-<s74eG0_+t6Lzis*'
    'C|lXmc(zgP5VjT<plwkwOuzu0aZ7@8XPNs#wi~i38`A*bKuK_-6lA@Bt7za{(;!pYX0`yZ<smB#0sGBJJm!+}!d5I_Aow6J0Ky$a>O?>AmYRgN'
    'Fyx>?hz1C}j-YKh1JT<39zn>uY-FS<yA%ut$myKLpts~8+4dvnLyehQp72&<CV!@vA(%;!k+sjYl-6b#bw39lX7tpL_RJqV=eSf3rt>*+w1MbN'
    'qy+Tx-vGJSkwk=^Fb7Aoa0c92r$b`45?Larqeu*QJwn2-4~eu5*7d;IMj{(Lf$DTvU8~Ai%5y#)4BKSs%+Vb~Vz}!e5WGGhFqrid(Fk7)5ZMI{'
    'L{dPQF{*stKoFXBM9~OgS3snGYs+$(a_Y5ckY_o#jKNB6q2$iHM}zli>n!I&Bohoz;~653dtT^Dc4P<6keI}L=2(C=*jb<m$DkPPd(gVW@<X3J'
    '6p*&2%w!vhc!(*wN9oLfn35bCw`|GGWQz*mf$I%jTgW~DB3aYt+?flC7_E5$WuTaiL{_tgT+&0Atl)&>4<diymsGFte_Z|&$<~ZY!NXL#Gaj71'
    ')IAF3%%Kk!)iv+HGfRG(EOR_18w{_(;Avn8a}b>ljEU3WO~%NRvCthY09J`bAlqYaBtBGFfWiXwA+Z`K7c4xVH5Q0wr<0g%?E-_sLiYvph*rrW'
    'et`bGu^_8~VA~Z`W5KF?GwRF4@?91zD4aJI$&7a=XP=Q?YlqI2$+`(pJRe`?w9W>>%zQ}B=7@d{2t?-0xu;&VkI0JZcs$w%=yXoqNY)@jpVC-J'
    '0CE~kGOSYViTf3sDdiBsSx%e%KK<i+ze~Tq2uZToU@Rn~V=k5LJ0Zy|c6bcFKc~(C1QgZF%#(u&nLv9x&Mtf$^jsphMYM2W8;4%oo57~9H8iWj'
    'A!IWPCXai5M2sZN?)XrCdH2YzaQb&qrA#E^F(kSgi*Tgz`dW}MCqIK5Pi`yiGc$Pahkcmkbc}jX>l)^DOz6R|MGFT%1&7_HuhkT%b-@s}Ay_bs'
    'u8agJF~nYp-mrh>wQjn)vd*Hsb!-8_HW2-UVi!1~=iXfG!Ki|@j3o$csSg6dNJu6m0Rhs^gBz=Ut1kA+mvhLuIQ+@4M}LahiiwL(Fy;XQxguQZ'
    'J0Gyh66hrd?U~VXe7I}H1dl#>amv%+t(sN2_ecZb5qj>qwpoAA+K#DAvtbtpoNz2D;>oMary>_mi$n1`SeL{&O{m3T#}LC!U-x_Ordz;p!ciq#'
    'A`1cINO%gpoB-Lbn3W;1p4r1|Yc(vV*y=JXFpONZNv=b&6naB0qu358;v)96CNFnn^vQc(*W@129l?^1;|5V0h0&JlJ2I8#yBOTcCi)TB*W|eA'
    'vPNOJ<9P#5PmY4EK9Bobq!=iOwAks5$x|xRyn0h#yGQ<}cjc##uP#nE2i?c5v(Ce~zeNiIY7pE4Al%_J_$f{&Z`G&@2;K&PGzhSI5Fl;6B@+VJ'
    'AOsBpY7l5;oB-zaKuT-OIhKmFH0(f*MT6?64;ITKqBXoF1b5bzh&fmjPuuo?BP3GrqrQ?c_JF9cAs3xLgXHo_9wxsv_s)9nr7d6@O@&V>P3y>i'
    '7Ms_qqVra3-L?VZX#c6-T(xXgtV&12{g+1pZz>A@?A5EQ^4HTzL}wb)q|~IG6LCMoJ|Vjx`l9b2g65H5M}<WoYsRH)esmvXCmb)9Q|xbEWPcd|'
    '*x!8hD9}nIfOpZVv}+qa>g^}J{hrzAa&&S~%-JmGxqrC{$>jZ@_s<yF9U6IEAYsoj!YxK#pD0JzZDX|d&&K{E_WUN*W`W~iDog#y${H7(&bp2O'
    'z+01E1&NWJl&cXvzE^Wvl^YIvgQnzJ?8zp4zZ#dSw*=w1Wp3HO+ylM5owD03KoJiBs2+utG)MK;oGzd^QUyc_lzqkAQ-#Qtfu|fu#oC66ecyim'
    '?iZV@uYR{bu5yQtf=RRM*lH>+7$ggWX5!@_&epFMXdl6<FesDCER`^hg@Bkl#7!mcTm90n`&x259}K2>lF4!yQF1m0d|+l>7i~#vb0kx+QuhP}'
    'Z=(?VnJJ<si;NsiFbvTR=WeXPbx2!FIY?k*kVe5J5fTO~tjv9iVH-_GKm|eo%smf*FUQ}LbCAa}Arh_&1rdyBU6A7jm>PxQj@weQD2lTW$m>JF'
    '6pg_~!5_LJGx<Q-rQuwvb>yOWwECwD5PS@X;jWu-tLT)7!*sQE;802|&10eTM$rPu_i|C{D<ME{5K9$__L;|t2<M%P!nk$jC6R0hdXt~!k}S^S'
    'o)>Y5%6bF?yRFFvW3@3AA;6=4c0M-xQ9Jcn6bP4GURC-*v}#I$#{Ew<0;BzQeOge2f*;lzpF0MbtlB{c<1Ak9d})>4j1H;AA|j`9N%ZcA=20?c'
    '!GGu7&tGi5`R;Eo&uaL&^T9MB!+2(22LLJnk^um&NgkHeezY(M3WF+y)U%^AhLV^WLoZ)7BRPQqXobmXMP0IxU>k|v_`Vj2o^QQ-v{EG4LL!cl'
    'kS&U3(LWchz63`U$2EkSkhh5}9MW_gP~$*r!=b3pCJuQI4u+qpmzjh_mAlpXwE;oqv^ogm8V3uA;ja4}k=F+Vc&lo*4Wk-B#PP|<jx`SHtZ<mv'
    'oFrN}z;qmvmJo4uIAGi+>&oyr6^Lfny@cM9x3fNm@LRSxg4D?%;-YHqL_}s^@*G32u~qkIEmQm4p54Rr-GgCQtz`S6Fj#T9K<c6tekcZ2P{D}Q'
    'PwU_1xu2;`sV5a~vt?2SWle-we5}w*-wmNepBg3VT1Bc=G-gkg_}-$}dxD^hR>>9wd=vs5nIG0atpx&a_QMtk;!y~Uf-kHxSv=%5K@j~Gtotx1'
    'Ng$ic$@dbqa*W=k5PRK_yGKT*iU(i1GVRFw@ZDcs)<MeW=P`@SZ@qOVM0F=bK<C+$t`_&YTHI(i!?_2;H{eWqwmED3mTZ4)L{@!G5eAbp>Skp;'
    '9Mo7nGE(+1rqkstO@V^#ny)a9%6VThnnUKE+toqxQT~HH^`LuZwKYUI@6K&$%a(u`$L2aLc78WvjsYEeXFvffghI?}D<%`hVJX2PVDte8$!u8C'
    'xTz0C_TIV7mauf#6A|Z_;mEkPj>xz*m#D^w!4vemT|zF!({2DpmP;0ivcE-+=Lx$>6c|h###)pJ&6uE{&{DVD$*DI?z_Dl*Bsa%O^813=5!Qjo'
    '>OhF$jE?$s(njy_2m}HVSOdQ)PrNhkujoYws<N5us+!@779~oA=bn71vzxeI`XHK=923`00v@Rl$(`@KLMY>H(E@_^00D>6yt={H)|kEw=Zk|c'
    'vu3=7gVL#l>&a)JgM9?7`uP=P_k?XUoDFdt!!B*-e2LtFZc!I>i#onVi@H<sf!R&JP2=k$AT|T}6#8}Jawqr*any9Eu!e_~l!HWPT2@VrcCi1F'
    '$pZa3&srP+6b%x|(dv?4*{q68<EkJrl|?J$*z-D;U~X};Iq2IJ?X&gulfZ4VfS^4<1Otd(fm9jCP}D~ZYXc&?Z44F?;W!e*T{i_HBxVJVqWNn8'
    '0&wfhvoTgBp~Q0$!eF#-(?TNYw1!nB{fSa+I$;teg@R~Kaw|Z81Pw&_pzk@RyUUVR<(AFcU|km~I-x-rZumeq*K@~!wrH4@Y1~-D;)B6{`@YCg'
    '+TYm+D57axreLxGQZEIznAVsuz%d6y&kY;SZ2QP&8GT}t1(01H)dplAVK}!HLr81cK7i?WOze+tf4J8{eaOBx0%y1V(y!IpJxqN0?N<9QpZDdh'
    'oZaq4QAX?NGfsW^+)G#KVs&vXFW;F-Dv9an_J@1zqSKMGgO%()+lnBEqnjV>bmwFzu`9e=3+pFq=7nsZk8Xdo+3w{d2cng1pTP{VBsMPnEXF||'
    'jwdF$A9(GV?%r&*n4&)YskB!ejG-J9c#M*h;%b@(Smhomh8yfZ8v9>KP{k*+V?>I)Xa77cH!ZHEeU>@IqI8NH?;p!_Zs-s5=xksb^ik{C#pi1M'
    '=-M~hCi58NJunQ+pIs)>wE@hH$~;b;p=6suN!bC&F}i+$_3%*Mu)d+Z9?ke-O0WlubZ8ju4vMrA!ANGVT2s5y-@Yh3(W6VMyrVb7Vpux9o;D(a'
    'f>qnml<N#ph=hKmA$el*I&V9yswa0a%fr=}pAG^#=q24F9M^C|n8$(lpwLg^F0BEmXCqh2vr7=qzY5UG;<Zc$m4wpbz((XCj)&V2tgKcZ&}=1='
    'aykmbaLd<<LsrzF9nHB7%~<vU2+@&q{l1ff(Al-l!B|zK*>AzHA+mFOa7e?!4PCp8-6RhfHQYLXW+>B4Oc1;V0XX}tnoT7{m3@P2;{~?;3E64E'
    '5OdL*!Z41~&ec57jQQLZ2GN<UcBI+mqYw-?e61TZt_1^O12V43?A>t;bSKJsDjY*v6(0m|x>mCHM==;~xYL;c(=e6&+#x_`j%>)P64;go-F|o8'
    'Wo%mauLS{0TVx9XoF)&kKdPX@#5<o?g+Mgzr|fAO548K;-3UIoq49L~eU)Hqo`Nv#Lg@B8f56x2QqRnJf7$ChQPjrNjHYNa&k{5Ere64Z@GKY='
    '9GLxSvb3j^{jnKuY;v*tI7H0J14Ui-*N?U&F(OKPXI_a@4XkEDO$Ewe_Y_(7)~n#~37N7_1NUnC%&%u&g|ihSRDP5wHi0MT<yv4U))PliZpC2<'
    '&-oim-p;nl7;O|#wIWSD6k1@ISBC<cC@5Pg1$G_@^n^DT^q@PpmdO=?x5<(RnD+pH2S?m_av}^I=hY1FDU&~~YmwY>0EU~rHn?I90Dzf%lvQJx'
    '0~}x;0bU6JY>qt5sx{1^{qvglZ+DZ+x=JO5?1J+{0LrYUoZ^|DfGG+gZL@U%bn;<##Yq@afB&2iLkt53Xr5VRocRN$xy>M!Z-s%u%)-h#1toLH'
    '^4yL6B-qPA>mC`GV2<Zo+nva4OP+NiN=dg*==*J`d$X^v2VI$YO}S{1Y~x5Lp)?TCp;-lcCW23b;KXyVldT=A&JR_)3*R)r*#+H8NM-0XYrX30'
    'nS+hC?^KoXVo{4B1UlM{y9f@td3X9Vh$pzMPZk#bC>EogHzi75ZpgraVU1{%F1(gAEyHm%hC3gjfontq(M(v&DnF-1<JfrY#-_ND1$p9`XIu0Z'
    '0^ukEqwS9PA+864kgNfK=v2>o1OmAdGaO9f9%OE<=E^Xqg9`?<2Ln9RVze6!xvFb3m^q7rK{$%RaKnQtzPzrGqc>{V1Oh$^!EnDr2v)@gVo;eU'
    'Ax?*2#04P$x>e;LSW^oGf;0_+Zok)hLh@?TP|TLAf<ZotfiV)gg@sl%(1zGN2`UR^KO6L0WHq(!Ze6iwD=keR@JAsS?stFJ*|SF=Sx$Cr(h&O)'
    'po@t@KY_apl6PMCfct~*m?OKG!wP4<!8V&GA|J<MxbK~&m>5<#^(~{-_e0EOw+D$~T1vO;Yu%G}T|lI~CF`C5zXu3D(D|aS{K0hs5x2=(K)_KT'
    'hTCq%gs2-J7Pem>6v0{UH-Q3=LNVI+^|1Zs3fUrB-Zz1Qjzclr_6Q1E9~5P%v~`oJ{GCHskb%Vtr>COZChH~zz5y0!V3AfRE@&G}aS``m5fAc3'
    'S3YW#ic&uFw%x(+(JS_8o4q-r5d#s#*ta}cfQZB7|CNl@byjKvHiNVe0x{fn<C^wjdz5}7Ym^mWb>>Bf?kE(aeP8QV_bW3%g3T4^k3wN|_#Y^s'
    'tGhqEB`0v7wwXsVGmELBRV;MQ5Sq-V6Pi<+u#i<qDOlL*COfA!Z>Mb#Tpto5G_$2)p{Z?h^L)RrqNGJYNfI{F<Q<>)V3d*_YD&$p2MZ%tK4>Ut'
    'g;10kF_yB4ya$SOz*#W_XM{vJcc#d;34&U^A}`XAf^z9X7>f$Dpa7r`{6hI+a0g=#Xv<&#p_=z1k)t+qrjGnG2cL54C5(A+!8tEVS6ENjTi);1'
    'x<Xa!-WCEs-~~!PrlaZeUD0*L1s`5~^}DY=S2M&;-or6%!IBTEgRe$nwC#Qa)0EQckYHO-27eTY;ieB27_9;b+5-ek;E2JlcbP#kvd>|4Nce5G'
    'ERKN3kr?i}F2BrTgg|sLuMY^fjg~?c$U>rZ>mA4|rIM4szLSAwhKQQRYNOEYd8k0F@6HfQ-1PBtvT+#H;E582xv@2`n~DX>$N^SXg9qN~|0pQr'
    '<0uTaJVaq-^_b*s2X1uJQRs<C3X!6xm38p(j<?aeDWczl0_ge!HU{Rv+M4(vY`tY<Mc4yGFKGm^G&qXQz}mrhln_@n9!0DRGov)(9u#)I7|uvy'
    'oF~|Uz@__R8Mn163{u)!(lHu$r6@*`56&1Ij@Me%9g(aMuqse%QP?p;BMO68y^O-jn%#-IVA<L+9>rj|<K7M7#*O07zTux)qgPyR6B(h|@laP-'
    '83>rS!ITO4C<wzXA26p@zzTVbR>&f`Jt)MnOz5{wm)nCpg~YfrUWjIzb@E3+7;X7lUl}XXo^3Z|f;}kYL+vkj3KXnLQ((WfRK|oh2)!ci;fT%t'
    '5Z+Oz=vDMWP1a03^eW%F^gl*18hYhurR{@fO0n;kZ=d;__DcS<SFf(hU-u*N`=7qp{NVkUKf47MQ@r!e>g*s$z#b&nK*AeHz)g{ObM@@gH+O@>'
    '#Fo*NEFf?X5OE?9ct$`>3fEZE>jekegF`ZKAOnZ6wxdDWYWyTfggr=P1BqZD!7Do&=&jz6*fe$0K#-dVL|hvXV1~85$|j{UqdP*cg11d~>2<;B'
    '9%Ul~#zC(I%vs0!quV-5aH_P@78KM8AU*Gc93z)~&T)U*!?~sfaWK8qUL}>*NKhgp!8RGm#^;qdXCR>suV^}Hy_)7to*~B2S2TTTc#sg|sOf&&'
    'H)d5MtVLoN-bt@Q{sc~7$f-BDYcQ_*T01P)Vsub6EBm~P_6r8dLF0F5?N2W_h@H`626Tai9e5bzme>EQx0~xi*)Oj4kLo$Kr`Bo&B2Xx88-zu$'
    'X^Zr>Z6hS-$GQFpul5<<Sh;Er#%Wt9<+$~h8K#2%Ega)Kkc@p*lU#OkK7iKdpK)ewpH~ro4Z~>9*9NDo2M4)7vt@dyn171{sII%*oAgPg8MVu6'
    'n^%K-L<AoNVYuVN65{MVw_CPsFE!|$L@UUp|9o$dRpJuS?F}et$N0GY7seWt3DJtJMhRYxf0tV`m2P7FDX1~JGVC7vwXdFqHDU<)OtML*RxDXU'
    'LJff@Br5KkXhgj&r~axOxoA%O6BOV$3d0>=8~FfhL4lpsnOaa#1{X1u{(6005aToE6#8vhyJvYFZE52^E>Kc3K%h9NLy<S`ky^=lJ$#0%b7Pa0'
    'E$CHd)FB{H1L1jfKm;=##;fe814#7KqdIib+K>p|YFI2tzzujJ8A!O*@dQNM3!!qPS|~!VYcVr2rhbi9hXkx<o{4E@MmoCv(OyT=VJ*|+gB{vZ'
    '`C$k4?~V7rmLbt-CWd*H7}jn-PJ8|9yCoUSWOA?4!ww-~T$48+5qxrySmNFm5@`<->G0)n`>^+&A~A8g37FDxpAu<>Iin#RXmwBkZNai9(jGO^'
    '40Kw`Q(hle*iD3dvh_$X?~J(KJ2Rr429e+f`wzz~iu2vSTov|F&^kjX;Y<J>)g4%t6a&crXdT-ZjX4_k1E`dT;F2JaUB9(j%!Mn!4MP){Kpy&4'
    '25NxKnndvNI2^;Rw~ZJRIBq70tqu*`MzFw0d%zg@-iU*P&@W#e^2?f{+oD8)c}lW@1Vhv7WnYxP<BGISuJC447*MTBTQF#7WQW*vV!PJ(+&FG?'
    'S|~!^W{WRWv(n&l<`ikr(YeP2*SkYQz=_rrrw|Rw(yKu=Ds4d^BU%%WiHUPZA)9eLP#JT<W6DKK=-s2KG{DZ#pQPG7!<Fa}Z05dzN{$mP(na4i'
    'Sli#1v}@byR^obb+hC}Xphjl~gy7lrd+K$atnpY^?^Spl(5%)4HAC|d62U-%)`mnjBNV{uNKnj<yz94FbTNe(0pgKa^XEKH*KgUn997URP~u3L'
    '@;K?KKk=dDTnn@0kXBZp=%eL;0~N7rPzY|+fR*}LE*%J~s$Ivl1<R^ZvAPz6LEcxhV;<D57Y~D56$aUKNrQ^gwHWOD4Ap+glQZ}rtqFsYBy92l'
    '?!f>JKG<c2*5?CnTEBl{41%pFs2E+K>^V05zup=m@KBQQDvH;=$qPZ_fEa>jJt)t+W3vKR;ezIDMy)}`?G8XO^1)SY!V<3z3Xzhgkc@kfhz3)j'
    'W(x0CheQyA!eolH2a2SjfT*E>^)ZD9bDku`ya$S`TjnL>eB>3L4>Xm@<ZL9s8Az}pH2wORV$b<Vei{_JZC@LVvMMOj7EPvrJxEv`CHYuy&2=z^'
    '-3$g2?tuafC@e{_zWam890&u6ZUBWd&PQ4u6w&QDAbV6zK;tSkq$aFSY7%V8x=It@019s?SnJ~oaJG9Y#pzJ^WI%yy)ViI8gsE!gJl3xg4%&6^'
    'o~<uOEa29>KeCO4QYvKE9yAgwv_Ht1gXSQ+8$jU<rl2({-DE{Hvfl%RH_pe1D`Fs8q5UD}!_>O7p=-(x6Z?btF}r8$6Pht@lXaD5+yey|P~Zp('
    'QN$yzFvplowE}g1OsV9}81uN6H}%x+?$P=Tki131e}y_9B*c&%^-FP#$!fc2M$C^epRQldw1_wv*<+|itA*$sI8HpM+=Pfc=!VffTS;OmhHW<Y'
    '57z!m#CNp+^73J{|MJYQWdCXF*xEl@`%jV6aR13Am-BG{)w8s|JOC@wfoh1_As~_g#OjJs>_jnC6VzHD@(74h*$fH@?^al-X!SQhH9xHZB02*I'
    '9syCF(fU*e(@WK4gHA+(3?y)UXTubh$-qH0s6mEAwP)t6aFAy!NgoW!Od3PX*8Uk`EE<qg9Lvb^EUsh!l&rWXIpePMfuUEWr`vb$8SSu^3Jlsh'
    'u%MtlP~<_LOt<mFoy^w)25wWfzz8-N{W?8k^2uQQ!-_KK$ZVHIx+o_GpQsUpZ;h1tYa{7=JqhMkaX0B=oCbt=ywsqp&skwD&PRppS5KdnCBFZo'
    '2MrvhheK=vJD)thY8!t$%8i5JlUJABa3%2IYJKwQv*)j$RlBI=gsj``>)Y?XxY|5_@wk%MPhMSKK6`TY?CGPj_ueZMK>uhwo%Nv_*IK>49;vC}'
    '%RZ^m$J*W8C)Fp{^qF0NRENEmyEmn|G?mJVW^R&|1~khCYZZP+Gu@cxjwQeO;_>Fg*YAGwP?K!S^AX_*sLr+Q6OQ1#A{-zzWA^TfFPf3Q(eHl|'
    'iu@RBZ(*N&@Ubo=8IX^z_AKwH_lo*T>i<PgH`%*`{g9Fas_&9MO$nRy{UI;NJ+u)2_6T~oQS6~;q((*P)5u~YXnxHYC@=Ne(@js7=qXGJ&Fq0S'
    'rATop!G-?jgtn762PaAHRIep$Y^JfzA=I8<d7RAH+;{^#bCY2@z!SLXy}2V3_8ytb_h0_>#U4@>l>0oiV_Msh=tD@8I^4e0q2pYk+Pr%5``V1R'
    'BI>{Z_Q`t#mx6i^u`MwlY%Yr4l9u7?eDebc-gzY|oa5SKI~1|p%&TAE05aU1QcS4sGM9|20AVpY$=#UF(p-hrrR8#M*N*PoTzj_XQ2OD!zq+io'
    '*5l#<MU_z(s-vVy!_YYGmjLnRY>B1%V@zmBWC#brFfDTR@6?iS1VvB60LG1qT5A=Xl;#4rSNt5Ju?*;l>g8r412f-Dg_=X8VMI?;ONm6HtIJd#'
    '1$|42Fb=a1^S<`+g<S0j7!%u0n;u+KfsI4s+W^M`%2Z67eLBFN+183~_9HnIvmBye$|1LiDo_Lcop(QfvH9k^zrCD<Z*WZm+X36gTwjVqI$52q'
    'Ia<CU-4$^}tnP8=Q}SAq7P$}05f|7`LN7MbZ*Y_nXmdpL(q>r6TmpMH&X^n)qdkwAPF`iuYBI#}1#>%|JD^e2zv2|8ASwSvb>)a6r$s#YOp)*2'
    'Rz4s>nX-2Hum;(Y3wL0F9yv9fnxT4w+Z&T%h4=tDFQeMaR*$Ln<aqx_CaU(4veT-7+^1S3P|j`<9^ZTISJj|Rsn`JQ$j$=0$+THW&daqk4Yr!h'
    'tTnoT84WUV>^HebGMn1kfxySC&z9%Vbd+3%%;qwG_xa_;#vi{q`*rdITG_dzX?0`8<o-w{S2D@d*bynp2*;=|c~2>wMm4Pn<)$HIpU_yEH5r6P'
    'm@Bz%N9Z8vH$XIKd!(tVKNXe6P2dde$<MboCm2l!DD9dJ3@C*rD`R+=s4-4Yahe8=AckhVA)`W$*`JiuqKzv3)Q1Gx)N2a5=U<iwsfs|A?y2s~'
    '3=WLTNKEAgBMX7@l&h$jb2Gl!V`lK$fb%J<px*6>oH92@lbMvADfH-I10XkZRObkz2_!z;_t=P{9pRM}<t-4Z8-cV{;9^?L>wf_7ilC$b_e9A6'
    'Jb9iA_{~9+D+f_KO~h_u!(T@8gS@vF{VlOdH5FSq85(qU(Vs>;sVOCGd&u2MV1>)lLV-QXu-QZN1V(5q&=Iz%byO15itoT!ok*p=rCicS$_F2q'
    '?;yZ`OQxu@o7~t}F7kjJ85tGw;f0uDkALbo-2L3Wq3#;8p}pb)X$f<%Rbk(9xkb=gVTKG#VQYJ=j$_NhvS7GVMPu|DTd@^`Urx0-;D6~8=^--b'
    'wtLSYKAJ3%8#t$h;-h@CXkJID5)E9swQ5SK3Gyw2r@Y9+sU9tTWJIQ%anrJx0}I9UOxliQrj$X?LUC}0x1SAz&@XfupmVdFel6Wz-@Gn5<5=ai'
    'OqAlNgz6~MH>^)aFij*wfcZM=yI*XszWUuDj&=hWK@-c)8%ZCbL40vyk4Vv$P}CtxZnHXu3M@rDHL5x_LWnh((}P&FOE9NF@R+XMRQG}DB>ypj'
    'Qy%<cTGi0#oAA0TN?B7%hM7mL6?3PCUPDz*sP&|nv}x^fps;8v$t2j&d9BFBY-Pofj|&BQj%oK3bi0;%P=<8MNh^0^xdJQ{*=LToQtn!kQR|}~'
    'vyrY<T4~+^;0w7{+U;mVAp53a#&A0jm&&aW?QPZ@;GiSir1e1n;Bg|_9DR^uoJgTJZi9%C^lcCq$f=!r*gUC3Lr7#rfI?alxBaO!ibyE8sUkAQ'
    'Y(SU6Et!krc<OauQK>T(khEVITr-T}1Vjc5TG2-g8cM{7aX~xOc2}7c>eN>vFkW_H)FmjKfL?iLB`oe#7&5ozU&H+29KByg&a=CnqYuhK=8#Y;'
    'd%o#dI#gCLERu$>SK&J%kQ#b((05Pp+Hl5FNS2oiuXOow8?dNK>Vi*3VvmP~-uXV~?(MYcdK2t)gd+K-q2SPKmT9RxW3hK`$-QOpLQ+wlVnTu8'
    'SqJco6{6fy6KP2fieyG-R3ee1Wlr3_M5KnGq|1(cCl-dTC=3f;@d4sQ$WqYhi6MEdyb%M0<?<8vN}`%7t#yc=>Bw@5q*M4rS~`;sJc^7rOE<~i'
    'W$D{qJ+7fNN1YWL+M0ra(-vvT_e?i-i#Bs#92><gR@xaME>%?+Zhha^Bqtqq3fYjDVOa$)P?|`g*Re+zA!{WHD2|;vv$!C!4Q%`C$r}JGcxqU3'
    '5Ee3y_mV!5U@gh2lvzXR=Vg*S4_`?cjr{!(t(q9fOI_<CQ|4fZ+PX7KV{oCGNFqOr+@xN4=9H~hj=Z2X%891L8yhMKw0zAz8!2<_<3d-Lc^OOt'
    'HFz`*em1nxNRXB~`e_ayuvY$@==`QOx@1P3gX07yFW3dGE_+JEmTw31a#@qQ@N7WbU>NMcc;?_OPe0GM{iYjtQ_N}bF~+uC(QpT?KW%8c#yg3h'
    'Od%D6Bj=d4b_mTZ=aGZdbGV13)Rr#84q&Tj+!sCYKIkG>T=X1_{5WSg96bQ%dk8NIMY&`S>de!UtFKwER29TKsBqlCap~BSs?T@CU^HSW{clf9'
    's^PprZ9A2^(TJz08x<x4Na&0lQ?dM#QAMl@_GWpQ$xo(GHw{S%4gH;yU&;@b`(@&Aq?)iJ6g8nuOn#)qxn)k$5#7_=i%EFUJw$ReDok?bg#yqA'
    ';vf^6h?tvTV-tqe1kJ3b1zeBe=pK%7j54f{vr*SkZjX3xM7JL~Pz5@Lsgx><qRbkn!6_?taVxC*^#=iuej-d1Ox`JuW5|}L)&*MTLO<jKi<nD?'
    '6dkvQQo5*FN`8?SjBDoO^1P)``(rX2;bDl$h0{<Nb@Lf-WP*X%IN?si7@9hf!HPQc)^!;DlBu)@gu;Nx%#@LvX>mipCK)d|1)iTzIkXYmhRgj@'
    'yD4@}OF3A{>e>WP5E;*la?~XZWXPiw7i$A|poC#6(S`#&8DX15h$rf|(~+u^mK=p>kQy?bm{eU(2x#<<5TgW6R4R@x#Nb%&e?=+-qme-nmWg*B'
    '%pgEU4^&$Jss@$OYN<$WfqKkF`SAO?n9xmIy)}DK7eyjx_&-HYUdsImljusi`KM7MUhI+uHI&1&kz+1B7P6B5rQ?&)ddEe77Y9evB(hq&Ju-SD'
    'Cy4TVx8M|-N^sbah2HRQ6uT_le??+NX7WLIoRbr41%e=ia9{4~jyUZ>B_MB9!zh2j5~b;ThyyIi93J((?Ak!!;2<C_B|Y|J=N^CAEy+4kU7{zN'
    'CyVR%4!dw3OpzWCE_~q2=s%rM!)#=FMZ~2JQ^>aqR0+v3h{>*-XK-6=q_S-m@>~$YtRgfjsd7Oq4yO}^$f`k8y@~Gd)U~u(Jsl%j;k9&M;wpGC'
    'Bts(IgoHR7tq`5;W5HAv06KJWcho6&lPH(T$c*<3RQc?s?BP*T*?XOvz=?v<9n14lu1cTXjkBsca&?!yn3K^U3sr{X!sPC2(QHz!opaf$;Fg<s'
    'nZ8~(A}(qUKm*c<MB(0u6D7RZSh|r%3Ah-wIqm6Rs+H(8>I}uxY9#|@DC$yK(YNa*(2=Z-@pi83C@mo+y2I|G9X-*Y$EYVwk(R8z;-d-1K@^<Q'
    'Ku)(0EIq-=>TDeEv^iu)5a?E%AzFFukZ{R%5R-haEtNWy8#x($0mwHjun?7p&uvSbf-r@qlzx<(>j9Ta#=6<{+)}E-xG}bE#0HcG&UEJkQ#z`&'
    'Ly9~aFE?aJ_uH?wA(4$HA7GdSa;3$1=ciUof1UVCrS8vA2{ZZSZdp+aKk!HKD)gyWN-63Lsw(v^EaPJ|Rn`M!_kRhK_N0A|(b$s@;KMx~9-#`A'
    'WK7<eI}l3U?+MP1P)LTa7LBN$9GxZ0QAONNqD~KD$=<32;l-NwYr>A=h|W9RfHZi>mS18q^^j&Z$7xAtU634_T$NEpGA?<2V@x-qQVZ3t3Ut`g'
    'B`-A$?Er<70VYbQ4Ox-<MyWD6zhq?{9=!}I;?xm?5UsF&ShOPoWc+}_O;FUzIfP{C!p<+5U?PY3MpUZ91jNZ|or$neslovYr?YADQBNgsl#N8;'
    '1WjOSvAm+a6@j6&N-wTl(o<Z55fnhcxa7N^I`Ocj(@1JXGBhj5D)c$?HvOYX<mrr7pRI9Wv{2842F7$#7>!0ADO+<$7HjQuaPZ{~=<iLg9X1_~'
    'Mkjz`0Ob4Y^K&#cqN6_>`R;5CQxTJjPSc^jBDG{*t9&nx42IKCL<8J|-d{$`f|12h`gI!p=ZZsa$tEvun%}0TQqRos&POwrwwOxg52W}`c<Bw{'
    '##{onbitUGj0hbd-lQ_>!W->MT&i9%->!p3N31Tj?WVpAw5v>u`tWSBEedsy4;K>~?lDof7BZ&g$j$<lv=|6rbkB}%No`;c)h2<~{7_xkBt|R4'
    'g6)R)6j@so>cF&!s9=pnOflYZ3HT(@uQPv~HENPdwcCB%M`emk6L}t7=*Who&nYfkGO({nW<y!b&Z<{4Qcxl<Iiimpe1H?;LJg5o&p!%Q0xr6g'
    'Pef#Lx`UU9N2`&%&kJ%MA8*;#QK;jQ*68)PRH_m4ZKZcjt`1k05>I1Ryz0fiFv%u$$!Or>4C-=IL9*0r%gGdaonQ^1`LXhaEJ$EZlL<6xMfB(!'
    '&*bqW0Fo|`%wEs6aKe|17A)<bvp%9SK}lUUJYHrYuT#;kd1=*o0**C>L6=pr5ua|`AlzG}aZ8{!#2~g^M259<31CqQ_tYg?L~32pmqx{g{*6n`'
    'f}xmiE5R$0jf~4Sw`53i!nh=B#>uQ{OepEtVK@oUFR6bYAFJCDsH@VLtY(nZjEG%`!Z?vahF}fU%GQi$ztoJE7-q0#S7bDxIHHlzD*IScQ*&`L'
    'g*w>{qfUeZ7kEKu0OMo=budV@93(5{eyJX%h<wAya6_u)w=gE`0l6jh41-*gIz4d6g<#|eaw1x401eDY_h`sYI@}(E!JtHoW`s|qPUnON2Ub5Q'
    '^x+Ocoj|Hi?<%H+(X?3XJX$7HA?D}a_wU^8c9Hc!b+ks6l!VR&<8B!zP=|W%U+6|vkLu`_jJjzoP2zNxy<?+zJA0QF%vbkTl4SR*5Ar0^bv8Q%'
    'wi;El%lDmLRg<X8l_CVA&1!KTE+k%ayv-46)13Q>V9p!SfBCdukV`eSCUJV=uXl!Ti*QjtHTh%2Roa{;j@e*CD5Hv&b)ru+(<>^oh7b%zUM$au'
    'C!<okJQAbP1Op%~6_PAXD35d<+V5SjiPkC70S%f%_RIFpya}+n_8o#Vh%9aW7wi$1Z$zaw*i$L`HlXxNM)Pk(CCUY<gnHm)^&(xMQ%gf3^NlFf'
    'r38?T3U5(}+>-M0hExV!H{y+mu4hMg4&%gBxrXv&RJ>%vVduf~$?%Q3R+^IGffqA=(H4t%qtj$qpx%n>m(=~GrearphudV6dAXC#CbY%py^-)!'
    '24um4;`LAp_zm`IY{~1Y8UiE71`yJ+!?Zq#{DF5jAytoe#cUOzL78T7YFf!?7iq+jOLE|Sc3%7dZng~_n<8(FfCiU%sSc6?H_;tzg{`e*>pF+U'
    'c5-wZ2RCbNLpRAsW8@fM3d^2=8&at>6R-v_`V#TT_dfovqf!UtG8#QZ%gqVP!s%~DsIJ~9&G#%Ry`-Xdb3%0`97LlQ-2;1|g+rW^LP-^oMu~~p'
    '%2x!$!nH8yZ-k~kMah9R8fVd|_BW@KjftGj>KRIWqT-^OP|~eYB7l+VLrY2}v)zbEp0<?8vrz{bREhJvh~aVn7h*PwbBwMDVhg$Zs~c)a(?x?m'
    'kx}WKV@k^giVnLDQxcsIoPMuX@h!-ImQ0=5<IZBnoMZ61g>Fh<baIZ_jZ1>(*fM;47)+IEmxLUBnK)<Uj6w=`U-MA{qnt1}EAiJEy>)yuI^upi'
    '6LLl>jr5GZx72p<9gYFS7*EroRSs`yei^;jK1-|bEr+m_gUBwUvP5%7qz*!PXN-NJ?DC!Vj1hIJDlShtquLpf&~V2MJ1cS$yQnjq5RCK(Va@qt'
    'Cl83+D54D^wo#)#Jqe77XJVAkk1*Ib)zPz>Jf+fsmG!GMI)V#yyiA(I(UaK`rmqOL3N#U8NOwM@zM~3^G=1H~u*Bh(QADOwrZT>gT>sHJ1nHhB'
    'BTX3^J;I7E6_yoW`zXU8wtsNJxTw50-X_O*7e%xOWJyJ$SU1|4;V|h0?J}1zEp+AY8KBy_jpQagDqITj<f79lbfN_twd&ji5AQE~q9T!#ah1tq'
    'LlaGHkBvJVEO1Z<ECjPl#^`i8PP&y!DYsmv)5oOKR46m%Ct9h{A{$fSd!me8$Pf%IFOZ(<?E~RGbZ+XXLY)qi(CR0(So6{8r=zrIPhbr8^Vx_w'
    'i7lQ_U~sf#>S($5Y^1m&`V1!|Q#VET&L~fl&oZ5c%n}^eQznZ~!<wxSm-GZ39%4`t2OM=G43}9&RzrcM@**J4ir==B>wtdo^l8e)D27F0jgzPh'
    'fy?YcEyR<k8%9TDc@z20Cx3~m%HUJhh55-_Q@SHM&UY`sg{llh+O_7%C@cmLvoPHsRmSLr%tpJCOa%9Qtt4_#x~Dij3fKk9V9$ul_kurKIo{E!'
    'EthQMiU9#Flem18vRuX_l*)eX@?d?Fpd6BAf(fI+LKaEM8{=hyF;#<oKE+hnj!s>~UR=^fJ0DMXJq9OZC@oOiZWF^YVI#(xJn)Gog|(!zG1Dqo'
    'HP;;GH=dbEsy6RS6w}CzM+zq*Rm>2|a>p809XjhkdkS}bpj(=>8#4wy!L`)C`E*Y->9%chCJZIYyOi$7`t2w)tY2r0rSk;iRt`7!CGFKmr;|$t'
    'kVYT+^67LhPPvXUtyLvvqp=yQrVe-D`0c3DowgV)yQ6Vj-+#NhhSIE^ZZ2n|h~5XJcwhKsXV|{cXagl03-J9>m2|^_Y$T5eRpC^xHOR|DAxitO'
    'i1dgeA5w69=f*}BtJ_g->&%AcqS1@0G<R6u*f1&5<vrP&J1$nayJw$sj$+-X5OqBy2tOLl`x*0+4bj1uMB*LC?U|+0k*PDQC<MchbRP5llj+OX'
    'mQrFc=2{wRUNMa^-HVmBBaL;S+_Obm4*s4<1JR7-KWGf1$yWNQ%BY%EO)#x+V@D<ES$xiDkma)~8NDAev#6S-7*o$x0rW=3i(9ffz&o7QHMk;_'
    '<igO!!XZba`mKaV@GetH<ZDvQ#94PvVj<M%R1$-`d)HpE8=%VdG6vHcvp!1HYeCl@r=zjeGQ3f~lt_2l=4jk=K4s5$H3GS>zl`5+jxNPyjM$S6'
    '7+qh>{F)QQW_)2pA0Y;9?K)$+2DlvHE<#=VZaKM0P`=XZ+`lTh!iaG>zWW&Fjx0TRLQanndUJvC9?4PyIYTxA1_;^l{c3?b8jX$vWRwD+=R4M&'
    '3rS@H0_$XYmJ2r-O<Y~@_wqJtvXaq32953}E-lIXN4ta+XO1iG!)Q1~4H?I+L|Ldl|9B=&p)e*&UdQ#KGa5aXa9j3^<s@1$y&Xwp)VLcKt%x~?'
    'i<30L^!7z%3dB~Hu)JWaP_>x{w$S^XR%Ey%Q(qAo*eIApj7#N;-IPRaohLAyd!-`_o)Agh%EFwJC^|lddys=Mx}x)8&}9=|_gnzNOnjHw>U*ur'
    'j#8~YqgGC#T!nPUS6fWXWg=fTI<y)?CQBvEQZ=~QAD7P!dAbRG$&*8C^wg65zbBEHK!hE!)~PT%qaC#1=k7z*OISP>R*U6~+D+xlogDA-Q0y%g'
    '+mQ;!bYd{9%9zogGTF5FQ$h(hJ3TPlffbl8?GR1`ZcC~*G>zU<BF59=ncOx#dNK#PY~(DlbMcO4U_+8_LtxglQ9?e2C7z+UQ?l;XQrfi`#erpd'
    '^*zX$%n8NP4qE)BV05p8yx39T-1o0QduTa-z%=-cO>oqesnJ;k^O9N5N2miA<kHane=FB$Fop}1a>l!V%W?&mB8o6B@R4&|rePa>(m5?u55S3L'
    'b&T54j*509qqCieAztL*+=w{iRA-|BJ#)T?9DGqya)Lxok9~~ha2T<fmUKTKPIi`Wb5R6#lPNdQ3Ym?h15^WP@9>3|CKeb#r{a_;Kn8MU9L&8F'
    'rxS+dnl+M0<f`-c(B$z(wCgsh*jy>2qy*%pJo|Ym+O>V}!MM9i{`g%e<2Rx;Y1g&YmbkbT7pnd{;D_kS>nP7EH)ywqPPdROsZ>Vj)KR+k27)!2'
    'C05|I7chZ!QWs<PsT2~%GA>(B+Jvq$YM9yQaJdll!3JKF_#hK_rlft-X2VmE+Ys-k#Cu1YuJHyos;NcUa^cRC!ggfoB5%Xy1t=o;UQ#1>l<6XG'
    'FeY+|a+8-C_jm)+bgcwuBh>|B_Wqtp3&!{sUC<_^QNB+$)3fTl_eOpdao~IAGhL4$H?~pA>r@TV-I2zLiIdZF7@{$Uq4XVICZ%d`hC>NyNAzqY'
    'EE_((`F`$U5!1Kc{k$|=eOzaKANH)2Zq0H#IneX9*rF9EI6h55L`N$cs3ubj&YVw0Yso<ueZ|NfF@{{1Y1ntrmT5$h=-XwGvA*=Ji8~9$UiM5?'
    'n!uRARh5|bt(DXa!_8cp8{c!fMn_|wlrUNRSn=-t7l=^&6y$Y{lTU0+sRE|FlvmPkZhK%_s&ybU8jg)>oW&ix>pPNks%~^f{~QR6cT=1{89Y5I'
    'I1w6`bIkYQAMeOY8bI_``wv9AKe9keLTYY|Az3NFJ*mtcY2GR?HVlo@2<TK-TuZCAiLx`QX3_gQ8H98LO0|uYFdBwPAWrjZdM1;lPi<hsbo1?q'
    '<$L5qeR7cubwMt|bPx7XN0yP1n+>%J5SJQ`%`win+BiaF9pg{d&_EA&Tw1;($Y^4q&83Xt?g^q8fTGU(6Jy*eJ&B0-aeon2jWdj9WiA*k`b%W>'
    'boIMlZgnq*6P$XjQRG=NQmoRroebSsF4&|EBrIK|K(`rkt!rpB?0KoiYd6Y9=*ZL!_(C-L9Z-mPA9{awu4GK=X7{CFDqBuz=kJj$ohd73k~MFM'
    '{X+2{N^V~43Nx3MG%;IqBMebl{1A9gH*`yyH7p`pA&6!2%WptgFhH~F;>l$?rP$1J?}&1xD;%sQ_ddr{>DxI6Czr9#9*YT$!bgzXXu%BR1BYoq'
    'CJ8x2WaMg<iNjzB@euDJQM!;7V)5iF)e(q|X1EZNzhke+F^UsAdFMJ0MZ}De#R_6|-MOaL5vB75+2}kQ77C8<>pMG{+-RhfX?mkyDwrQ<(yI~>'
    'oY9n&=l6n~Oi<P-t42Z|-aS3pkwegJ|AgDvTg8QoX%mod`%-fm4^LS~^-?tR8kh3N3M98RS}UT(MAXtp#E+3V7(?cJ3PpCLX&F?8o*G%p<nNiX'
    'WW;ZcyHD;zw@@$Akiy(oosEY7dQiMC-S!vqyo;7Nn`onjd^8%hm)Sdb=P|9xRO`Hh?6XywlIC|$-aiLoGPzoEWh3Ak+gTR(bJE0qv|3tq6;ng;'
    '1d;B**Kq?%wIBuSjeBtMq=f25AdZq%vK-Wjy6d;C`=UFJM)4&(H7)bbNz)peNKd1Yd{Q};cc5iXr5cQma1pE-5~^G5ZK<ZstqQI?ij8h@XpHa-'
    '=se#q3qUj)*%_)*@Y$cHYQ3TqlhHB6`D8RDlA8{~J%Wg4Ss_Lvr8WrMe7~ewg3+4G8W{538@`jN*6%u?F<@AvNm|HK3P>}k7Me5bbk<BpYBH7y'
    '&-XJiq$Ujl=-7OjDr$slX^|OrlgZM}tJ!E$1bQCFot+!cjHR_08siOuIhcVLGJFYVmP&NA>(Q(ft!g1iw4}3jKm9{8(LpRTsI*U*L$FNY2>0P^'
    'ZOKwMVnj50%0*JiTz8(lSJ7Brv#v717_7>BcOOx_jx5k!dC*EKWYuE(UZ5-CJKg!X<Y5?fexe?SdlJ$uX}W?dSv@m&+<J`Wj(wbT%e7QyO(hsT'
    'azPcj+%bM$mx$%$Ia)!cdwsR?!gMe;5*y{hxt-y9_XoBjOWzbZ7(LrkhLfCE$Wg2PawwuTxjg~i=S$6P^MQdV31H1?2Y=6$k)|ve!59&8E}t-r'
    '-O<ac)ZYc8Q6I3s&v%TR3~itrLOwXSA!fR#gImbb)3@aNjn0MybpHpJJcVS<mQBWNi<~eoWWybB%^!(&wKVHwTTDjKHHREdG-5?)E!mCH&{I8n'
    'L;9dx&UyK%Q@et#5p`5rS_d6X5;fk)>Ms=$_teF_B8`BPk!lD=^rgVLU<LHxeSRdLeEInG$Io7uPkjGJ58hl|bu#1X?Pt#}A6>q^dQ@@GfnE`1'
    'l~+$5J$?MPeCnI>>CGR%|Kjr}7s5yHzxbQW=N-NBkw>rIyfiAouU<WV^z>Er0m0-^2SHmH{%}|hUcT?i+gC4Y%-{WDQ$l!U#?3z}+<dk9&hLNS'
    'eQEhz`>kc<wvV|sq?+%nen0-woFRsXoxHEl+12aEm(L$vT|BP7SI){8zq;H!dG~jltJkjt)ZhQ~{g?lGb0x_9i}LB`7cU+?xu`Jt6L~LBHc#G`'
    '*Hb?5#c!@QpS=6|rXy#&`TB+Y_TRn`sr<#~mlyj?*v&tE`*_og^!e&@`EvQy|5?BG&DHxa|J{pCua(F{p{=}ym;G=5_~KR74|&6|dATigz7YuI'
    '06lvA>7%D_<z;<s<8%LoRhxVCgIABPp1rv;Ufm94^~oLF)e&ge3vBihCYBBU@%t}-w)sah_{w=PIID=-k9U8^BRqx<(buorWsEWWxJ?Ko+&!+n'
    '7k%FQhZiD?-hcTwPbv|+`9z2}`R)L_X~z2Y^H+tH_p{Tt9>3W9uivP@a)^jf@Wn4bE3EL>o3DT8yVV-<(w|%uRuWQ$=dRy>@!KK+Dwcg&xb69N'
    'Ghoms7hnDE@#crZJDZpCO7<{`JIEtP7-PT>X!Ao)whuYs+ZXT35ixEq+4vWqzt~)sJ8$^U$7}+w)WjAY{>NqVA4Cqdd*=X?cyz84M4}k}xJh+%'
    'KJK2@PUC@ms>v<bSA^B4Rk-HQHwPrDVj&RG=j7rD)KTn1_gCEQqmD)#5^iOv3$`Se3Y2CiK8051d5i(t4T?DIKJC5l9*5M4Wvca)5hJ&{$58>f'
    'dj}0X#>*8MSy=M2hI5~Q{8;|}?>;T?7Gc-{Rl&bQXuE07(B*z!G7gIDpi(@ZY<}?m%YRvY*S~zx%2yHVpMSRJu*8FhWiK-xI6*gIhyGzB`>>65'
    '>CuTha>e!zmUY8ghsnHP`^V<%VxGLLxo1yV#FLBC-v7yHz@2q=3KnuHI`lt|khqMI;lqa{Rs*8oVUyBYIyL;bDVK7Y+s7Phm(!5#-~IWgo2Np3'
    'VV;X;uQnGVi7qyuJgzS+Eyp{(K>qyMlgch@HvYWUFsyxUH{X=^@Z0K&mlgD=l7?|VEo}L6bN!-~^)`djVR1fqs|UE!1iH_D!Dhei4S(<5Z_a`D'
    '3aD4F<wZOZJ*@`+H<$Ntv)A{q{Jy;FTDg;d`?9;hKkoF6%L<mFHx4fAp|U4H%Rjr^^x?hLbw2g|58gE&dDm_Hee(z3fPY|VdxlUlUOATfz<lJD'
    'M;2N5u#L@g9WCp>^6Cp~@!qs0*=IkX$Xq&h#fC(w)e1uCvZ9&c>(%Rh_xVm(?35Q{=ZZ@@S}Iebk{90yoB!|s_}4m3WLu8UFUqC$Z#Hj=tdQIA'
    'Rr&E}oATzin-5#QlP4d)|KdMiZC=Xfxli4bi&u@XsV$VN!jYfWC!_xM5z+f3%`ryXz0V_-Mg1cB92ael`$n+1>zML25-gaoF5f-CWn5wqoi14C'
    'm@6jJgB<Rj(~683aN6i~j~HykExQY>a9EK}B2y|p?Wimq_Np*O`NY$A|9SK!yMy)QtIs!Y-hcTYga!XoV`-F+{A%;0aE{!4zur6-l}wXZe)rSr'
    '!N0hrO)_|k(|GHn%9rx`^^2#kDj&&@<aIu}dR#Hp<67Fv6;aFFqG9%_elEQAi<aQ(_`DR3`?Tncm0DRe&F%=w=lYOW`@4AdsDAO~>;Cs|ekal9'
    '5B}#r{mwVPMhIL9Ot}t(^ye=M5W<l!p1rJSeDk<M<AIA`b4~<;%U`RbQgV$vtM%>XH@^DKPl|h~Ao=F;lZ!|7`T6+S)uSIi>$`>gO`Sr8p60~;'
    '=)u*q%L7*YlgCdUl{Zp<U-fP(e_7Af(0R*xPNCLUwsQ`ZH~ai;eLNbS{Z0LC|M)w9QlE_B+rE1J_SsK9`bkS}b5c8c4`?6zXzuRHA;q=QF{HRs'
    'K4y^ogL2BBT-40}Ko%?Zzi5nd{x))wKFFdyy!*U{_~qLtMTT_mwJ6x4M|}ML%fAz4qE#8+fBApaAHRB4eZ0?Gk3N2RA^(uKq2h!`A72P}US2%D'
    'EWa(}fBz6SeDAy8{_w#^|J8cYDoG@=yC;pl|Dc&8()2!VMgwCVA;CJriB0=Z`Lp@CZmxpOL`G~zamt_A{7KE9q4|@WKbv)+SqGbSYJWiK`cuK*'
    '&?jO~PV$JUc2_A(cF`?y%6C4y;hCx3a)UEdyXX36ruez4m-Dc>Pd;wqTk=+aEwAVYwP$Q#!T<Y)$ww#a&mMpBrbfXc4o4!Wat<ouT)I=@_KT`L'
    'zQ{i4hR_HpomyxpuKH*dE>Y{mA2?t;-l|U+PH=i!XX`fJ&wU4P=Xe^jcJ1jL-L4^|+g(+o^VE5`M&DWaQPp>PSo})Al|Obbgq<I5w4KVwI@(V0'
    'wJwRFA7fZ~a7SNSJh@ALDUO+?@(+)9>b+m|OTGHiiXDggA4ObDoAT^$A{hUPNSUjf%0#zUCJxVUrG;L7^}Err{_eZKm7jk2>hWizUs#z25BKZp'
    'AN_aFedD<|wK%9Xvs5c{scy~-RHr`3pQ-t?`c&1&WV)V3Q@!!#RdCkEi79+%c=h*ckDbrCzo*uLKiE_4$X`M(b>8jKcUIAX(RY?8!D3?Rn-_A!'
    'UsO)FBA4XK|7}tDL~GnUsXUmKerbxRZ{I{X#RCf4m_0vQVF<)L>9t7~D#NKRg{_@Mb;(6tOI1e;>I#86*}YB?tlOG#^>6FU_&WEj&gQCpfue7<'
    'e}-CU)a67#bp<&zX|K>!jY6(d4xmYpgr>w4nx0<rCp3Rz^Jnw#>NHDeddXnZbE2qq&7Y0&=o;hEwV!K@NY@&tjS(r|H(jWONim-PS|r-uSkV#7'
    'Yjl!}=kw~Kb(8+^?Pn!LYxXft2Xu_V3;F1LgH<yPExutCYOh1XxRnnuWgW&CGG!fw8Zu=Sjar&1&1lx46IA)Ths4%_g(0z3fMKz_?CbCTZu9&M'
    '?lUtYGZ?n_(SMuKJ6jzN^|c30{<r+uURvP>mqa`es@r=6Z(*Q!QJ*1q*P*(xyMgIWPNK<RYdDVOQW}eK^LEyb!aC#pb~n>tJjla}D?a*fC8~L='
    '+pdmN%0{0(`k}DZQ@LF0*y!$sy!&(6?GI%$wMPEz>GQ&G<@<m2cz+zSxIzEzi%On68B07RQz~tHO#ikDFMj&^)t|li<XL+%9)0}c$;BgKze*s='
    'p}u(cpNdbeV!RrparpG@`!9Zdsc(kOAN+CkPd~h<h<Ld9@K3kj`ud+#H~FU*FA7%mx4^b8j(Gm~GlAyQ!f{RV9q=D||A)Rw5@^D^AMTCh_M^Bz'
    'd?r7B{@G>_X{|jL{RnH(d<uAdv?`!0rYPXPth^qY#VRIg4W0fk%6HU%E7#vYub2B+j>hH(?>=wcHa{pf^#-<X+dGT(@%zz(r*B_Xr(bxbN&u_Q'
    '<3$l+f{rhOX!r<<w-W}B@X__-a()Y2R!7%|>i6~1PhSY1R#)%t+2yP9u=zKi3b~Iis-SiGhm{O{RT%t{{GZ$ipVf4}C`Z0J2w1Hozx=e80jd32'
    'Bt!Wb>#zT(Z{;1=*GCNfk314_Q)qwvSbh|VRc*d3b|qg?(b4R=dQ|Q5ZRM+%S6GoJXY9?ZT4G%YnzDN_S)V+*ss+~=(+A((%g^LCAHV$IVt0lb'
    '3!*1dF5dm79IB>Z<Kor3DlGB!&e*NqxctAuOi$l`@yoZ3L=m3Ud{mwJm(@S01X`0|uzT49S_=vleP3%jZ3=Net5^K%4ZUoICBJ;w(pS#qZy#?!'
    '@8c8U?8}GUN%`cf-@V;!YOuElBW4?rR5;Hf3AW+o<`0FahsC=8%~iACr<Eo4)$hv9_1BLp7r-xG48B3}?%?YRKMua`vYhzM|5Sb7L-|Se`{%{P'
    'dH2PW5>756>eX(SPfN(C-1&b|?$G~Gj`y$2qScZW8N(wo`0m(!b@j48sokpbzf0NKi&|EfOS#=+z25GWJdk4|FXY`{efG5y1nWeEuzZmTmAL%)'
    'qF%Cu(f>}KmWX&4Hmnxhow5gMx8C=f4c6N3cWWKs+mByXi~XRuU~8y%4Dhh@z5VhppU4aQ;=i=NEsmbD;U_hY58JvVk&VB-l8yasEnAyd-P78|'
    'ds&w6%e+GTFF$RmZDhcMA_Q7;|4I4ItJm*7f6-9f?Q!#i+SSw$RDZEt4)VqI*MHh{dn>AP`Su|#zyImu`Uw7g{l^zofVbRDU1~s~@ux-4ke~mb'
    'C(UmK((>{DyTa+qUp}nJdbN4|xVz6TMX&whzg*VmroXr2!mD^!e^#HX|M;>##1#&kKM=<HHx<0q?>W5r2mMzmhx($&<ga(2RjS~p#YDNRt&Zk+'
    'H^OOvLWAU`{7Q7svT7~GzNpXSUu=H(;<C>cqO^WiDNgmvd|d@l<YwPK&;qzh6Z!nA2;yc-FDt#I4$u_dtEiKQKf7up1ND}wDpxr<#UUYt7SiO0'
    'zbQCZ?v47xTd{ho*H$|VuUe$4RRw5y{O&iKKmNym^gnDHc~Gz0MHL#Ix#5mV+I~_+BtEW;m0HW{xl(GasJ)RZC9Pb{<xReN^s*#<J+5yQ-v!U1'
    '`x|z_BkC;M&|QbzJ&X0TpR^W&7y)v%mOl%!b*@<d4j_HsZ$o7#^#55+W6|*bnOw7#$?~J0>@@A3-uk0<gZ;a|3L4dd@L@kT|D?LYYCW(3E#K`s'
    'fqhjH*IMc;JF&Xq24b~FeeW`TjkA)g<u-&$j|wd_H0D!nRsV_nKGumzgNyWWsiG8KdDZA(oq<qY$W0LJr&S&8rO1%#5FE2qwtM3FMJY6W-_>_F'
    'Lu~_BK7>k*=-F=-9AcFGO_g)<mzOomHN5p`k2&%Ey7;qFl`B^<Kd3Cl$4{O-dsAmXVrh>av#WmSo<FFy=c`w*A6-AYc=4pt^>+`SeAYfGtCb}Z'
    'P@cO4`I%Idu6=p#5}wn{C445j-4d6tx<#mTQk+KzWq_GW#OT7zC9=mjWr@$8z1g=4dWRv;+Dlo2fo3fcVl_4LwtFcLtA&C~A%{7P8Zd-1m3;0J'
    'm~hS#AteVmhe65;u1uVmwM6td>(GXr6Jv@qmf$j0A<cRtC7~#BaC68IAcDLS3Uk*8aTaT|YmDDY|6ojCCeJ5D7<K*{v+k#Ajflz5StEO97C|&?'
    '4JZNWcuPE5Db_BN7FlK{DiGxC%(@`jHNxyQ(Bb?wFwS3t=Hnq(<^pS^`D?`a{DEnKcbjQ8yUQ}<=EF)0uo?LU)}T3;H!!k8jA_;yDX=WzX0MUc'
    'Y&yuzx+Fx1W%9Dh0&DmM)_?`pzy;QrrMNp$&d|qrYh9!@%bJDqJmaoj<+kkFSrI6Dv)r9H%cQB-2=g{ntujw3Y1ZIb?^zVIx&`ozHJE|r6wg|t'
    '3{!TA=iot(Ft874)|+WoVK<W!c7B#$g<Jv(c_tzWc&2lP<i_)&#m_l2qBy(AJnN<?&8I_jejXjE48_SytU(K`5f)fumKy*9#7MKO=X#AaOB)G6'
    '<f-F%E{%GPSy}))^2HMn7g!_BU4!KG&R@g3`D?)3HHxN*^VSe5Ci)SNtTB&{Eb6NKXMr{10&C>?Yecue8Z>W>oLrc@m$JqzgSTA+X0O4s-2d#$'
    '$ZI!CCbw(k`D*|!u*L#-%sczd8Vl?N7r=CumD#Sbz+UFvQS};u7Fc5e{s;@4m#_dHyuiDi<v?uLz<H)}yUYTwmlt@w^RC=xjhRPR)Vh>mfi=Pc'
    'Ys}wEc42`v()=|%EwDzMzXtFEYb@|y=3aHx8h(K_V1YFj;4xg_z0d-f&bx9;OlPK}JBk=A_E=b8jX9V`jFdwe7IOPFk{20PT4c;w15#p$vsCtW'
    'jXCc&VHrUNX^tFA1OlkLZA`auUY1qLpb0mVlCp-My9R|>H(d$=Jos4=$P@zwH!mXFu7L}zK?|%g3y+-3l;~NG*lLXwW=XQLM(JxfmyS|^iSyP-'
    '#Pc-6>_piXc8?3Jk!G$@O0H+d$f{)`&N}|NateE%yN0}+Sqg+OYMHE@)9f{ppT#0_zmR}R3#`EltPvMjV;-h4ltwgefi-6FSWF^@{H)8rU1MgF'
    'OHQS`6VGzV_|kH%Br?J*C8Apf=PpxvK+HQi?IN>65Td-hL<s{fu?Q`($gEVgqP|0LG>?)%zOYJ|w}@~*;jEN-FIS(4Xz%A?5=HBk8)t5=nfD}@'
    '4$OPYrDt;}oe7p$B+pwUVDw?uZSFlH22U^#6B1nw=G{HANG`o5abC)J6mIc66O(2Un70VZ)sbfc8w(xD&!Z%RP?BbO;$@L?55hc=0zQ}M=se-p'
    'E;5f2k+E4s!90^(mJwEgIPFs2U(i#pgB0fhQ!T=?B7<cS(ad#)!HtsxtKtoxT%P%!>P2|gB3ObQFwMtz=q)wt(Cl?MMmJMjwu{WVhe~7L=+Vtx'
    'Bvu1cmsn&rw#_<Wp>^gl9!E#c&C`C`MP|OE2%LSMOGz%wGVerlb}k{z<Cch`RRYc1PNc{Q=GCY{OeC5q<OLR)r}{t&EEd(gDwlQ<oViGLVO~0?'
    'FY!m>5azLc=8((m^O=ievFqk;r(R@UA+(&n60n*Vo++b0iY7PD4r&*fS6K-Ja=*{3RBIQRR}_g(4q?c&#3EsdMR<ut;u4FbB^Jr^7h&gSFTxIK'
    'iACZPi_8R;#Q1kJOPh(DTwe3;z;+Qodl8(MPuMPkZiz);iA84mS=(i3_A)WY$aCLNwFoV-NL*r(yuc#!bc0fbgE`KMWOa-15{two7D-Dik{4Kn'
    '=Di*7oftmz)QffzoWBUoN+Xd)N;XTF$+6&MjhuL%M3H5Z2<v(7O#&a>?Mpw0i})oLna3$5l+3c!?3OrU3yMM;XI}0kA`ZQuDH4dGSWxp82@ojG'
    'RK40o!V-%tvYoWVB6)#DW{Syb5ts{XC_-*txG;##A3~mI-nNUxxr^}aQ|*U~(7Z)*<`RirVv%{2coZvpUJ0caUUMZ*F-u&B61piBYO@v*{jU^3'
    '&RitPy`E<^`)d~oZiz+w5{tkRi_GI#D#NT&n)!BwUxTp95{two7FlFFd4Wa1EwPASVi8zk5uCk9&I<sW^8&z@?D?6C1d>Cz$RhLJPSL6}#(BU7'
    'F<2bU(;mfaa&i?Xp1p{tnW{xuB*l3dMC~GZfkoySlx2|;aP_knVT8Hc@qq!;|37LNLJgOgP#Cbl>cYg#(u5j@key<|*PA4kBtvf1fZXm8pL}#B'
    '=s;~t@RiAjca^{dOh5uqHK1Eo;)@|iWkYVu1t~-kH9)8a-w*;ldmDV_wvxXK_*@#$mA619C=e`aYij`m?3MK'
)
raw = zlib.decompress(base64.b85decode(snapshot))
assert hashlib.sha256(raw).hexdigest() == 'd7af85cf84f5e8f32ae9b7584c54f915c563a3e768f4daf430ff62ac0422d205'
data = json.loads(raw)
print("Đã đọc dữ liệu nhúng từ bộ test đã bổ sung.")


Đã đọc dữ liệu nhúng từ bộ test đã bổ sung.


## Đối chiếu mốc kẹt xe và cấu hình tốc độ

Tọa độ mốc dùng vĩ độ, kinh độ WGS84. Đối chiếu theo ID với cả mạng đường sạch đầy đủ và tập nút trong bộ test. Mã 1 áp dụng cho phần Xô Viết Nghệ Tĩnh có liên kết từ Bạch Đằng, qua Cầu Sơn và dừng trước ngã 5. Mã 3 giới hạn ở nút Hàng Xanh. Mã 4 khoanh theo hành lang Ung Văn Khiêm. Mã 6 chỉ áp dụng các cạnh của D2 tiếp cận giao Ung Văn Khiêm. Tham chiếu các nguồn mô tả [Xô Viết Nghệ Tĩnh](https://tuoitre.vn/duong-xo-viet-nghe-tinh-o-quan-binh-thanh-un-u-615-lan-trong-9-thang-2024102110025915.htm), [Hàng Xanh](https://znews.vn/ban-do-28-diem-nghen-giao-thong-tphcm-va-lo-trinh-thao-go-post1692028.html) và [Ung Văn Khiêm – Nguyễn Gia Trí](https://dantri.com.vn/thoi-su/ma-tran-ket-xe-o-nga-tu-hang-xanh-gio-tan-tam-20241025090505064.htm).

Mã 2, 5 và 7 có tốc độ tham khảo trong JSON. Đinh Bộ Lĩnh và Nguyễn Xí vắng trong đồ thị. Tuyến D5 đã nối tới Nguyễn Gia Trí và dùng tốc độ cơ sở do thiếu căn cứ gán tốc độ kẹt cố định. Các ghi nhận theo sự kiện trên Điện Biên Phủ đến chân cầu Sài Gòn và Hẻm 356 được giữ ngoài danh sách tốc độ kẹt cố định. Bề rộng hẻm không có số đo dùng 15 km/h theo cấu hình BOA* 2.1. Tốc độ đường nội bộ cũng lấy 15 km/h từ cấu hình đó.

In [23]:
for marker in data["markers"]:
    print(f"{marker['stt']}. {marker['tuyen_nut_giao']} | {marker['trang_thai']} | {marker['so_canh_ap_dung']} cạnh")
    for side in ("dau", "cuoi"):
        ids = marker[f"node_{side}_osm"] or "chưa xác minh"
        coords = marker[f"toa_do_{side}_lat_lon"]
        present = marker[f"node_{side}_trong_bo_test"] or "—"
        print(f"   {marker[f'moc_{side}']}: {ids} | {coords} | trong bộ test: {present}")
    print("  ", marker["ghi_chu"])
print("Tốc độ cơ sở (km/h):", data["profile"]["toc_do_kmh"])
print("Tốc độ kẹt theo mã (km/h):", data["profile"]["toc_do_ket_xe_kmh_theo_ma_doan"])


1. Xô Viết Nghệ Tĩnh | Phần có trong đồ thị | 19 cạnh
   Giao Bạch Đằng: 12437015526 | 10.8028254,106.711403 | trong bộ test: 12437015526
   Ngã 5 Đài Liệt Sĩ: 366417938 | 10.8099275,106.7126155 | trong bộ test: 366417938
   Mốc cuối thật là 366417938. Chuỗi cạnh Xô Viết Nghệ Tĩnh qua Cầu Sơn chỉ nối tới 12313135292, cách mốc cuối khoảng 62 m.
2. Đinh Bộ Lĩnh | Ngoài đồ thị | 0 cạnh
   Giao Bạch Đằng: 10118771782|10118771783 | 10.8030331,106.7093775|10.8029343,106.7093708 | trong bộ test: —
   Khu vực giao lộ: chưa xác minh |  | trong bộ test: —
   Hai ID giao Bạch Đằng lấy từ CSV đối chiếu và vắng trong mạng. Mốc cuối chưa xác minh.
3. Hàng Xanh | Nút giao trong đồ thị | 7 cạnh
   Nhánh Điện Biên Phủ tại nút Hàng Xanh: 2030638872|1497276525 | 10.8013004,106.711558|10.8016017,106.7115512 | trong bộ test: 2030638872|1497276525
   Nhánh và nút nối về Xô Viết Nghệ Tĩnh: 2403042654|1497276515|1497276532|1497276531 | 10.8012043,106.7112269|10.8011855,106.7113752|10.8016809,106.7112111|10.80

In [24]:
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from heapq import heappop, heappush
from itertools import count
from math import fsum, inf



MICROSECONDS = 1_000_000
ARTERIAL_TYPES = {"primary", "secondary", "trunk", "primary_link", "secondary_link", "trunk_link"}


@dataclass(frozen=True)
class Edge:
    arc_id: str
    u: str
    v: str
    bearing: float
    duration: int
    congested: bool
    length_m: float
    is_alley: bool
    is_main: bool = False
    speed_key: str = ""


@dataclass
class Label:
    incoming: Edge | None
    duration: int
    turns: int
    parent: "Label | None"


def turn_cost(previous, following, threshold):
    if previous is None:
        return 0
    angle = abs((following.bearing - previous.bearing + 180) % 360 - 180)
    return int(angle > threshold)


def network_nodes(outgoing):
    """Nhận diện nút giao và nhánh hẻm cụt trên đồ thị bỏ hướng."""
    neighbors = defaultdict(set)
    protected = set()
    for edges in outgoing.values():
        for edge in edges:
            if not edge.is_alley:
                protected.update((edge.u, edge.v))
            if edge.u != edge.v:
                neighbors[edge.u].add(edge.v)
                neighbors[edge.v].add(edge.u)

    junctions = {node for node, adjacent in neighbors.items() if len(adjacent) >= 3}
    leaves = deque(node for node, adjacent in neighbors.items()
                   if len(adjacent) == 1 and node not in protected)
    removed = []
    while leaves:
        node = leaves.popleft()
        if len(neighbors[node]) != 1:
            continue
        parent = neighbors[node].pop()
        neighbors[parent].remove(node)
        removed.append((node, parent))
        if len(neighbors[parent]) == 1 and parent not in protected:
            leaves.append(parent)

    branches = {}
    for node, parent in reversed(removed):
        branches[node] = branches.get(parent, node)
    return junctions, branches


def enters_dead_end(edge, target, branches):
    branch = branches.get(edge.v)
    return (branch is not None and branches.get(edge.u) != branch
            and branches.get(target) != branch)


def turn_delay(previous, following, threshold, delays, junctions):
    if delays is None or previous is None:
        return 0
    delta = (following.bearing - previous.bearing + 180) % 360 - 180
    if previous.u == following.v:
        seconds = delays["quay_dau"]
    elif previous.v not in junctions:
        return 0
    elif abs(delta) >= 150:
        seconds = delays["quay_dau"]
    elif abs(delta) <= threshold:
        seconds = delays["di_thang"]
    elif delta > 0:
        seconds = delays["re_phai"]
    else:
        seconds = delays["re_trai"]
        if previous.is_alley and following.is_main and following.congested:
            seconds += delays["re_trai_tu_hem_ra_truc_chinh_phat_them"]
    return round(seconds * MICROSECONDS)


In [25]:
from math import atan2, cos, degrees, radians, sin

threshold = 35
profile = data["profile"]
delays = profile["do_tre_re_giay"]
nodes = {n["id"]: n for n in data["nodes"]}
categories = {str(row["stt"]): row for row in data["corridors"]["danh_sach_tuyen_ket_xe"]}
congested = dict()
for code, segment in categories.items():
    for edge_id in segment["canh_osm_ap_dung"]:
        assert edge_id not in congested
        congested[edge_id] = code

def bearing(u, v):
    lat1, lon1 = (radians(float(nodes[u][k])) for k in ("lat", "lon"))
    lat2, lon2 = (radians(float(nodes[v][k])) for k in ("lat", "lon"))
    delta = lon2 - lon1
    return degrees(atan2(sin(delta) * cos(lat2), cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(delta))) % 360

def available(road, restrictions):
    if road["road_type"] == "Đường đi bộ" or road["highway"] in {"footway", "path", "pedestrian", "steps"}:
        return False
    access = next((restrictions[k] for k in ("motorcycle", "motor_vehicle", "access")
                   if restrictions[k]), "")
    return access not in {"no", "private", "destination"}


def speed_key(road, code, highway):
    if code:
        return f"ket_xe:{code}"

    if road["is_alley"] == "True":
        alley_types = {
            "Hẻm cấp 1": "hem_cap_1",
            "Hẻm cấp 2": "hem_cap_2",
            "Hẻm cấp 3+": "hem_cap_3_va_khong_ten",
            "Hẻm không tên": "hem_cap_3_va_khong_ten",
        }

        road_type = road["road_type"]

        if road_type not in alley_types:
            raise ValueError(
                f"Loại hẻm chưa được cấu hình: {road_type}"
            )

        return alley_types[road_type]

    return (
        "duong_chinh"
        if highway
        else "duong_noi_bo_khong_phai_hem"
    )


outgoing, incoming = defaultdict(list), defaultdict(list)
for road in data["edges"]:
    if not available(road, data["access"][road["way_id"]]):
        continue
    u, v = road["u"], road["v"]
    assert u in nodes and v in nodes and road["oneway"] in {"yes", "no"}
    code = congested.get(road["edge_id"])
    is_alley = road["is_alley"] == "True"
    main = not is_alley and (road["highway"] in ARTERIAL_TYPES
                             or road["name"] in {"Đường D5", "D5", "Nguyễn Gia Trí", "Đường D2"}
                             or code is not None)
    key = speed_key(road, code, main)
    speed = profile["toc_do_ket_xe_kmh_theo_ma_doan"][code] if code else profile["toc_do_kmh"][key]
    length = float(road["length"])
    assert length > 0 and speed > 0
    directions = [(u, v, "F")]
    if road["oneway"] == "no" and u != v:
        directions.append((v, u, "R"))
    for start, end, suffix in directions:
        arc = Edge(road["edge_id"] + suffix, start, end, bearing(start, end),
                   round(length * 3.6 / speed * MICROSECONDS), code is not None,
                   length, is_alley, main, key)
        outgoing[start].append(arc)
        incoming[end].append(arc)

junctions, branches = network_nodes(outgoing)
arcs_by_id = {e.arc_id: e for es in outgoing.values() for e in es}
assert len(arcs_by_id) == sum(map(len, outgoing.values()))
print(f"Bộ test đã bổ sung: {len(data['edges'])} cạnh, {len(nodes)} nút")
print(f"Đồ thị xe máy: {len(arcs_by_id)} cung có hướng, {len(set(outgoing) | set(incoming))} nút đi được")
print("Cung dùng tốc độ kẹt theo mã:", dict(sorted(Counter(e.speed_key for e in arcs_by_id.values() if e.congested).items())))
print("Phân bố tốc độ:", dict(sorted(
    Counter(e.speed_key for e in arcs_by_id.values()).items()
)))


Bộ test đã bổ sung: 509 cạnh, 458 nút
Đồ thị xe máy: 660 cung có hướng, 410 nút đi được
Cung dùng tốc độ kẹt theo mã: {'ket_xe:1': 19, 'ket_xe:3': 7, 'ket_xe:4': 41, 'ket_xe:6': 6}
Phân bố tốc độ: {'duong_chinh': 171, 'duong_noi_bo_khong_phai_hem': 20, 'hem_cap_1': 259, 'hem_cap_2': 32, 'hem_cap_3_va_khong_ten': 105, 'ket_xe:1': 19, 'ket_xe:3': 7, 'ket_xe:4': 41, 'ket_xe:6': 6}


In [26]:
def movement_cost(previous, edge, target):
    delay = turn_delay(previous, edge, threshold, delays, junctions)
    penalty = round(delays["phat_hem_cut"] * MICROSECONDS) if enters_dead_end(edge, target, branches) else 0
    return delay, penalty

def route_cost(route, target):
    return sum(edge.duration + sum(movement_cost(route[i-1] if i else None, edge, target))
               for i, edge in enumerate(route))


## 2. Chọn các cặp OD

In [27]:
import pandas as pd
from IPython.display import display

corridors = {'A': 'Đài Liệt Sĩ – Ung Văn Khiêm – Nguyễn Gia Trí', 'B': 'Nguyễn Gia Trí – Ung Văn Khiêm – chân cầu Sài Gòn', 'C': 'Hàng Xanh – Điện Biên Phủ – chân cầu Sài Gòn', 'D': 'D5 – Xô Viết Nghệ Tĩnh – Nguyễn Gia Trí'}
# Danh sách đồng bộ với data_test/cap_od.csv; nhúng để chạy độc lập trên Colab.
pairs = [('OD01', 'B', 'Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sài Gòn', '5110361174', '1497313031'), ('OD02', 'B', 'Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài Gòn', '5778236281', '1497313031'), ('OD03', 'C', 'Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điện Biên Phủ', '5778236310', '5337055595'), ('OD04', 'A', 'Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí', '366417938', '366376129'), ('OD05', 'A', 'Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm', '366414641', '1276696904'), ('OD06', 'A', 'Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Nguyễn Gia Trí', '3631245426', '5110361174'), ('OD07', 'B', 'Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm chân cầu Sài Gòn', '366416151', '728690586'), ('OD08', 'C', 'Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Điện Biên Phủ', '366433763', '5337055595'), ('OD09', 'C', 'Hàng Xanh → cụm chân cầu Sài Gòn', '2030638872', '728690586'), ('OD10', 'D', 'Miệng Hẻm 356 Xô Viết Nghệ Tĩnh → giao D5 – Hẻm 42 Ung Văn Khiêm', '366418620', '10910534491')]
original_od = {'OD01': 'OD01', 'OD02': 'OD02', 'OD03': 'OD03', 'OD04': 'OD04', 'OD05': 'OD06', 'OD06': 'OD07', 'OD07': 'OD13', 'OD08': 'OD14', 'OD09': 'OD17', 'OD10': 'OD21'}
assert len(pairs) == 10
assert len({od for od, _, _, _, _ in pairs}) == len(pairs), "Trùng mã OD."
assert len({(s, t) for _, _, _, s, t in pairs}) == len(pairs), "Trùng cặp nút đầu–cuối."
assert all(c in corridors for _, c, _, _, _ in pairs)
# None chạy cả 10 OD mới. Ví dụ ["OD01", "OD04"] để chọn hai cặp.
OD_CAN_CHAY = None
YEN_K = 3

known_od = {pair[0] for pair in pairs}
chosen_od = known_od if OD_CAN_CHAY is None else set(OD_CAN_CHAY)
if not chosen_od or chosen_od - known_od:
    raise ValueError(f"Danh sách OD trống hoặc có mã lạ: {chosen_od - known_od}")
if YEN_K < 1:
    raise ValueError("YEN_K phải lớn hơn hoặc bằng 1.")
selected_pairs = [pair for pair in pairs if pair[0] in chosen_od]

def has_endpoints(start, end):
    return start in nodes and end in nodes

print(f"Chọn {len(selected_pairs)}/{len(pairs)} cặp OD.")
display(pd.DataFrame([{"Mã OD": od, "Mã OD gốc": original_od[od], "Tuyến đường": name,
                       "Node đầu": start, "Node cuối": end}
                      for od, _, name, start, end in selected_pairs]))


Chọn 10/10 cặp OD.


,Mã OD,Mã OD gốc,Tuyến đường,Node đầu,Node cuối
0,OD01,OD01,Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sà...,5110361174,1497313031
1,OD02,OD02,Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài...,5778236281,1497313031
2,OD03,OD03,Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điệ...,5778236310,5337055595
3,OD04,OD04,Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí,366417938,366376129
4,OD05,OD06,Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm,366414641,1276696904
5,OD06,OD07,Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Ngu...,3631245426,5110361174
6,OD07,OD13,Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm c...,366416151,728690586
7,OD08,OD14,Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Đi...,366433763,5337055595
8,OD09,OD17,Hàng Xanh → cụm chân cầu Sài Gòn,2030638872,728690586
9,OD10,OD21,Miệng Hẻm 356 Xô Viết Nghệ Tĩnh → giao D5 – Hẻ...,366418620,10910534491


## 3. Dijkstra

Tối ưu tổng thời gian gồm thời gian trên cung, trễ tại nút và phạt hẻm cụt. Dijkstra chạy trên trạng thái (nút, cung vừa đi vào) để xét đúng chi phí rẽ. Yen và BOA* dùng cùng mô hình chi phí.


In [28]:
def shortest_with_turns(graph, source, target, movement_cost, *, previous=None, blocked_nodes=frozenset(),
             blocked_arcs=frozenset(), simple=False):
    """Dijkstra trên trạng thái (nút, cung vừa đi)."""
    serial = count()
    initial_visited = frozenset({source})
    queue = [(0, next(serial), source, previous, (), initial_visited)]
    best = {}
    while queue:
        cost, _, node, prior, route, visited = heappop(queue)
        key = (node, prior.arc_id if prior else None, visited if simple else None)
        if cost >= best.get(key, inf):
            continue
        best[key] = cost
        if node == target:
            return route
        for arc in graph.get(node, ()):
            if arc.arc_id in blocked_arcs or arc.v in blocked_nodes or (simple and arc.v in visited):
                continue
            extra = sum(movement_cost(prior, arc, target))
            new_visited = visited | {arc.v} if simple else visited
            heappush(queue, (cost + arc.duration + extra, next(serial), arc.v, arc, route + (arc,), new_visited))
    return None

def loopless_shortest(graph, source, target, movement_cost, **options):
    route = shortest_with_turns(graph, source, target, movement_cost, **options)
    if route is None or len({source, *(edge.v for edge in route)}) == len(route) + 1:
        return route
    return shortest_with_turns(graph, source, target, movement_cost, simple=True, **options)

def dijkstra_shortest_path(source, target):
    return shortest_with_turns(outgoing, source, target, movement_cost)


In [29]:
dijkstra_routes = {}
for od, _, _, start, end in selected_pairs:
    route = dijkstra_shortest_path(start, end) if has_endpoints(start, end) else None
    dijkstra_routes[od] = [] if route is None else [route]
print(f"Dijkstra tìm được đường cho {sum(bool(p) for p in dijkstra_routes.values())} cặp.")


Dijkstra tìm được đường cho 10 cặp.


## 4. Yen K-shortest paths

Tính tối đa `YEN_K` tuyến đơn, xếp theo tổng thời gian có trễ và phạt hẻm cụt. Bước tìm nhánh dùng Dijkstra trên trạng thái có xét cung vừa đi vào nút.

In [30]:
def yen_k_shortest(graph, source, target, movement_cost, route_cost, k=3):
    """Theo cấu trúc đường gốc và nhánh rẽ trong notebook Yen đã gửi."""
    first = loopless_shortest(graph, source, target, movement_cost)
    if first is None:
        return []
    selected = [first]
    queued = {tuple(e.arc_id for e in first)}
    candidates = []
    while len(selected) < k:
        route = selected[-1]
        route_nodes = [source] + [edge.v for edge in route]
        for i, node in enumerate(route_nodes[:-1]):
            root = route[:i]
            blocked = {p[i].arc_id for p in selected if len(p) > i and p[:i] == root}
            spur = loopless_shortest(graph, node, target, movement_cost, previous=root[-1] if root else None,
                                     blocked_nodes=frozenset(route_nodes[:i]), blocked_arcs=blocked)
            if spur is None:
                continue
            candidate = root + spur
            key = tuple(e.arc_id for e in candidate)
            if key not in queued:
                queued.add(key)
                heappush(candidates, (route_cost(candidate, target), key, candidate))
        if not candidates:
            break
        selected.append(heappop(candidates)[2])
    return selected


In [31]:
yen_routes = {}
for od, _, _, start, end in selected_pairs:
    paths = yen_k_shortest(outgoing, start, end, movement_cost, route_cost, k=YEN_K) if has_endpoints(start, end) else []
    assert all(route_cost(a, end) <= route_cost(b, end) for a, b in zip(paths, paths[1:]))
    assert all(len({start, *(edge.v for edge in route)}) == len(route) + 1 for route in paths)
    yen_routes[od] = paths
print(f"Yen tìm được {sum(map(len, yen_routes.values()))} phương án.")


Yen tìm được 25 phương án.


## 5. BOA*

Tìm tập Pareto theo **tổng thời gian** và **số lần rẽ**. Bảng tổng hợp lấy tuyến ít rẽ nhất. Nếu bằng số lần rẽ, chọn tuyến nhanh hơn.

In [32]:
def remaining_times(incoming, target):
    """Dijkstra ngược trên node: cận dưới nhất quán cho thời gian còn lại."""
    distances = {target: 0}
    queue = [(0, target)]
    while queue:
        cost, node = heappop(queue)
        if cost != distances[node]:
            continue
        for edge in incoming.get(node, ()):
            candidate = cost + edge.duration
            if candidate < distances.get(edge.u, inf):
                distances[edge.u] = candidate
                heappush(queue, (candidate, edge.u))
    return distances


def remaining_turns(incoming, target, threshold):
    """Tìm số lần rẽ ít nhất còn lại, theo cạnh vừa đi vào."""
    distances = {}
    queue = []
    for edge in incoming.get(target, ()):
        distances[edge.arc_id] = 0
        heappush(queue, (0, edge.arc_id, edge))
    while queue:
        cost, arc_id, edge = heappop(queue)
        if cost != distances[arc_id]:
            continue
        for previous in incoming.get(edge.u, ()):
            candidate = cost + turn_cost(previous, edge, threshold)
            if candidate < distances.get(previous.arc_id, inf):
                distances[previous.arc_id] = candidate
                heappush(queue, (candidate, previous.arc_id, previous))
    return distances


def boa_star(outgoing, incoming, source, target, threshold, delays, junctions, branches):
    """Trả một lộ trình cho mỗi cặp chi phí (thời gian, số lần rẽ) Pareto."""
    start = Label(None, 0, 0, None)
    if source == target:
        return [start]
    dead_end_penalty = round(delays["phat_hem_cut"] * MICROSECONDS)
    h_time = remaining_times(incoming, target)
    if source not in h_time:
        return []
    h_turn = remaining_turns(incoming, target, threshold)
    sequence = count()
    queue = [(h_time[source], 0, next(sequence), start)]
    fewest_turns_at_state = {}
    fewest_turns_at_goal = inf
    solutions = []

    while queue:
        _, estimated_turns, _, label = heappop(queue)
        edge = label.incoming
        state = edge.arc_id if edge else None
        if (label.turns >= fewest_turns_at_state.get(state, inf)
                or estimated_turns >= fewest_turns_at_goal):
            continue

        # Cùng state có cùng heuristic. Thứ tự heap bảo đảm thời gian
        # tích lũy không giảm ở state đó, nên chỉ cần lưu số lần rẽ nhỏ nhất.
        fewest_turns_at_state[state] = label.turns
        node = edge.v if edge else source
        if node == target:
            solutions.append(label)
            fewest_turns_at_goal = label.turns
            continue

        for following in outgoing.get(node, ()):
            if following.arc_id not in h_turn:
                continue
            turns = label.turns + turn_cost(edge, following, threshold)
            lower_bound = turns + h_turn[following.arc_id]
            if (turns >= fewest_turns_at_state.get(following.arc_id, inf)
                    or lower_bound >= fewest_turns_at_goal):
                continue
            duration = (label.duration + following.duration
                        + turn_delay(edge, following, threshold, delays, junctions))
            if enters_dead_end(following, target, branches):
                duration += dead_end_penalty
            child = Label(following, duration, turns, label)
            heappush(queue, (duration + h_time[following.v], lower_bound,
                            next(sequence), child))
    return solutions


def route_edges(label):
    edges = []
    while label.incoming is not None:
        edges.append(label.incoming)
        label = label.parent
    return edges[::-1]


def route_metrics(route):
    length_m = fsum(edge.length_m for edge in route)
    alley_m = fsum(edge.length_m for edge in route if edge.is_alley)
    return length_m, 100 * alley_m / length_m if length_m else 0.0


In [33]:
boa_routes = {}
for od, _, _, start, end in selected_pairs:
    labels = (boa_star(outgoing, incoming, start, end, threshold, delays, junctions, branches)
              if has_endpoints(start, end) else [])
    for label in labels:
        assert route_cost(route_edges(label), end) == label.duration
    if labels:
        fastest = dijkstra_shortest_path(start, end)
        assert labels[0].duration == route_cost(fastest, end)
        assert all(a.duration < b.duration and a.turns > b.turns for a, b in zip(labels, labels[1:]))
    boa_routes[od] = [route_edges(label)
                      for label in sorted(labels, key=lambda label: (label.turns, label.duration))]
print(f"BOA* tìm được {sum(map(len, boa_routes.values()))} phương án Pareto.")


BOA* tìm được 15 phương án Pareto.


## 6. Đường chính giờ cao điểm

Đường chính là tuyến nhanh nhất khi chỉ dùng các cung được đánh dấu `is_main`. Vẫn tính trễ tại nút và phạt hẻm cụt như ba thuật toán. Đây là mốc so sánh mô phỏng.


In [34]:
main_graph = {node: [edge for edge in edges if edge.is_main]
              for node, edges in outgoing.items()}
baseline_routes, baseline_times = {}, {}
for od, _, _, start, end in selected_pairs:
    route = (shortest_with_turns(main_graph, start, end, movement_cost)
             if has_endpoints(start, end) else None)
    baseline_routes[od] = [] if route is None else [route]
    baseline_times[od] = None if route is None else route_cost(route, end) / MICROSECONDS
print(f"Đường chính tìm được tuyến cho {sum(bool(p) for p in baseline_routes.values())}/{len(selected_pairs)} cặp.")


Đường chính tìm được tuyến cho 10/10 cặp.


## 7. Đối chiếu và so sánh kết quả


In [35]:
def summarize(od, corridor, name, source, target, method, rank, route, status):
    row = {"Mã OD": od, "Mã OD gốc": original_od[od], "Hành lang": corridors[corridor], "Tuyến đường": name, "Thuật toán": method, "Hạng": rank,
           "Trạng thái": status, "Node đầu": source, "Node cuối": target}
    values = dict.fromkeys(("Quãng đường (km)", "Thời gian (phút)", "Số lần rẽ",
                            "Tỷ lệ đi hẻm (%)", "Cạnh kẹt", "Trễ tại nút (s)",
                            "Thời gian trên cạnh (s)", "Phạt hẻm cụt (s)", "ID cung"))
    if route is not None:
        distance, alley = route_metrics(route)
        prev = (None, *route[:-1])
        costs = [movement_cost(a, b, target) for a, b in zip(prev, route)]
        edge_s = sum(e.duration for e in route) / MICROSECONDS
        delay_s = sum(v[0] for v in costs) / MICROSECONDS
        penalty_s = sum(v[1] for v in costs) / MICROSECONDS
        turns = sum(turn_cost(a, b, threshold) for a, b in zip(prev, route))
        values.update(zip(values, (round(distance / 1000, 3),
              round((edge_s + delay_s + penalty_s) / 60, 2), turns, round(alley, 1),
              sum(e.congested for e in route), delay_s, round(edge_s, 6), penalty_s,
              " ".join(e.arc_id for e in route))))
    row.update(values)
    total = (edge_s + delay_s + penalty_s) if route is not None else None
    baseline_time = baseline_times[od]
    row["Tổng thời gian (s)"] = total
    row["Tiết kiệm so với đường chính (s)"] = (round(baseline_time - total, 6)
        if baseline_time is not None and total is not None else None)
    return row

primary_results, alternatives = [], []
for od, corridor, name, start, end in selected_pairs:
    has_route = bool(boa_routes[od])
    if not has_endpoints(start, end):
        missing = "Thiếu nút đầu hoặc cuối trong bộ test"
    elif has_route:
        missing = "Không có tuyến đơn"
    else:
        missing = "Không có đường theo chiều đã chọn"

    for method, paths in (("Dijkstra", dijkstra_routes[od]), ("Yen", yen_routes[od]), ("BOA*", boa_routes[od]),
                          ("Đường chính", baseline_routes[od])):
        if paths:
            evaluated = [summarize(od, corridor, name, start, end, method, rank, route, "Có đường")
                         for rank, route in enumerate(paths, 1)]
        else:
            evaluated = [summarize(od, corridor, name, start, end, method, None, None, missing)]
        primary_results.append(evaluated[0])
        alternatives.extend(evaluated)

checked = Counter(row["Thuật toán"] for row in primary_results if row["Trạng thái"] == "Có đường")


### Thời gian của 10 OD (phút)

Bảng lấy kết quả vừa tính từ đồ thị; BOA* dùng tuyến ít rẽ nhất, nếu bằng số lần rẽ thì chọn tuyến nhanh hơn. Các giá trị là thời gian mô phỏng, không phải thời gian đo thực tế.


In [36]:
times_by_od = {(r["Mã OD"], r["Thuật toán"]): r["Thời gian (phút)"] for r in primary_results}
comparison_df = pd.DataFrame([
    {"OD mới (mã gốc)": f"{od} ({original_od[od]})",
     "Đường chính": times_by_od[(od, "Đường chính")],
     "Dijkstra": times_by_od[(od, "Dijkstra")],
     "Yen": times_by_od[(od, "Yen")],
     "BOA* ít rẽ nhất": times_by_od[(od, "BOA*")]}
    for od, _, _, _, _ in selected_pairs
])
comparison_df.index = pd.RangeIndex(1, len(comparison_df) + 1, name="STT")
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.float_format", "{:.2f}".format):
    display(comparison_df)


,OD mới (mã gốc),Đường chính,Dijkstra,Yen,BOA* ít rẽ nhất
STT,,,,,
1,OD01 (OD01),14.68,13.21,13.21,14.68
2,OD02 (OD02),4.03,3.28,3.28,4.03
3,OD03 (OD03),10.90,10.18,10.18,10.90
4,OD04 (OD04),4.40,4.40,4.40,4.40
5,OD05 (OD06),2.59,2.59,2.59,2.59
6,OD06 (OD07),3.27,3.27,3.27,3.27
7,OD07 (OD13),5.08,5.08,5.08,5.08
8,OD08 (OD14),1.67,0.96,0.96,1.67
9,OD09 (OD17),4.87,4.87,4.87,4.87


In [37]:
import pandas as pd
from IPython.display import display

view_cols = ["Mã OD", "Mã OD gốc", "Tuyến đường", "Quãng đường (km)", "Thời gian (phút)", "Số lần rẽ",
             "Tỷ lệ đi hẻm (%)", "Cạnh kẹt", "Trễ tại nút (s)", "Trạng thái"]
method = 'Yen'
selected = [r for r in primary_results if r['Thuật toán'] == method]
print(f"{method} ({checked[method]}/{len(selected_pairs)} cặp có đường)")
result_df = pd.DataFrame(selected, columns=view_cols)
result_df.index = pd.RangeIndex(1, len(result_df) + 1, name="STT")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result_df)


Yen (10/10 cặp có đường)


,Mã OD,Mã OD gốc,Tuyến đường,Quãng đường (km),Thời gian (phút),Số lần rẽ,Tỷ lệ đi hẻm (%),Cạnh kẹt,Trễ tại nút (s),Trạng thái
STT,,,,,,,,,,
1,OD01,OD01,Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sà...,1.595,13.21,7,25.6,22,149.0,Có đường
2,OD02,OD02,Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài...,0.406,3.28,3,51.2,4,52.0,Có đường
3,OD03,OD03,Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điệ...,1.279,10.18,2,15.4,20,68.0,Có đường
4,OD04,OD04,Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí,0.488,4.40,0,0.0,12,30.0,Có đường
5,OD05,OD06,Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm,0.280,2.59,0,0.0,8,21.0,Có đường
6,OD06,OD07,Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Ngu...,0.371,3.27,1,0.0,9,26.0,Có đường
7,OD07,OD13,Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm c...,0.573,5.08,0,0.0,11,30.0,Có đường
8,OD08,OD14,Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Đi...,0.245,0.96,2,80.7,1,11.0,Có đường
9,OD09,OD17,Hàng Xanh → cụm chân cầu Sài Gòn,1.739,4.87,2,0.0,1,130.0,Có đường


In [38]:
import pandas as pd
from IPython.display import display

view_cols = ["Mã OD", "Mã OD gốc", "Tuyến đường", "Quãng đường (km)", "Thời gian (phút)", "Số lần rẽ",
             "Tỷ lệ đi hẻm (%)", "Cạnh kẹt", "Trễ tại nút (s)", "Trạng thái"]
method = 'Dijkstra'
selected = [r for r in primary_results if r['Thuật toán'] == method]
print(f"{method} ({checked[method]}/{len(selected_pairs)} cặp có đường)")
result_df = pd.DataFrame(selected, columns=view_cols)
result_df.index = pd.RangeIndex(1, len(result_df) + 1, name="STT")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result_df)


Dijkstra (10/10 cặp có đường)


,Mã OD,Mã OD gốc,Tuyến đường,Quãng đường (km),Thời gian (phút),Số lần rẽ,Tỷ lệ đi hẻm (%),Cạnh kẹt,Trễ tại nút (s),Trạng thái
STT,,,,,,,,,,
1,OD01,OD01,Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sà...,1.595,13.21,7,25.6,22,149.0,Có đường
2,OD02,OD02,Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài...,0.406,3.28,3,51.2,4,52.0,Có đường
3,OD03,OD03,Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điệ...,1.279,10.18,2,15.4,20,68.0,Có đường
4,OD04,OD04,Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí,0.488,4.40,0,0.0,12,30.0,Có đường
5,OD05,OD06,Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm,0.280,2.59,0,0.0,8,21.0,Có đường
6,OD06,OD07,Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Ngu...,0.371,3.27,1,0.0,9,26.0,Có đường
7,OD07,OD13,Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm c...,0.573,5.08,0,0.0,11,30.0,Có đường
8,OD08,OD14,Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Đi...,0.245,0.96,2,80.7,1,11.0,Có đường
9,OD09,OD17,Hàng Xanh → cụm chân cầu Sài Gòn,1.739,4.87,2,0.0,1,130.0,Có đường


In [39]:
import pandas as pd
from IPython.display import display

view_cols = ["Mã OD", "Mã OD gốc", "Tuyến đường", "Quãng đường (km)", "Thời gian (phút)", "Số lần rẽ",
             "Tỷ lệ đi hẻm (%)", "Cạnh kẹt", "Trễ tại nút (s)", "Trạng thái"]
method = 'BOA*'
selected = [r for r in primary_results if r['Thuật toán'] == method]
print(f"{method} ({checked[method]}/{len(selected_pairs)} cặp có đường)")
result_df = pd.DataFrame(selected, columns=view_cols)
result_df.index = pd.RangeIndex(1, len(result_df) + 1, name="STT")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result_df)


BOA* (10/10 cặp có đường)


,Mã OD,Mã OD gốc,Tuyến đường,Quãng đường (km),Thời gian (phút),Số lần rẽ,Tỷ lệ đi hẻm (%),Cạnh kẹt,Trễ tại nút (s),Trạng thái
STT,,,,,,,,,,
1,OD01,OD01,Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sà...,1.830,14.68,3,0.0,29,159.0,Có đường
2,OD02,OD02,Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài...,0.624,4.03,2,0.0,7,91.0,Có đường
3,OD03,OD03,Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điệ...,1.278,10.90,1,0.0,23,77.0,Có đường
4,OD04,OD04,Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí,0.488,4.40,0,0.0,12,30.0,Có đường
5,OD05,OD06,Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm,0.280,2.59,0,0.0,8,21.0,Có đường
6,OD06,OD07,Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Ngu...,0.371,3.27,1,0.0,9,26.0,Có đường
7,OD07,OD13,Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm c...,0.573,5.08,0,0.0,11,30.0,Có đường
8,OD08,OD14,Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Đi...,0.244,1.67,1,0.0,4,20.0,Có đường
9,OD09,OD17,Hàng Xanh → cụm chân cầu Sài Gòn,1.739,4.87,2,0.0,1,130.0,Có đường


In [40]:
import pandas as pd
from IPython.display import display

view_cols = ["Mã OD", "Mã OD gốc", "Tuyến đường", "Quãng đường (km)", "Thời gian (phút)", "Số lần rẽ",
             "Tỷ lệ đi hẻm (%)", "Cạnh kẹt", "Trễ tại nút (s)", "Trạng thái"]
method = 'Đường chính'
selected = [r for r in primary_results if r['Thuật toán'] == method]
print(f"{method} ({checked[method]}/{len(selected_pairs)} cặp có đường)")
result_df = pd.DataFrame(selected, columns=view_cols)
result_df.index = pd.RangeIndex(1, len(result_df) + 1, name="STT")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result_df)


Đường chính (10/10 cặp có đường)


,Mã OD,Mã OD gốc,Tuyến đường,Quãng đường (km),Thời gian (phút),Số lần rẽ,Tỷ lệ đi hẻm (%),Cạnh kẹt,Trễ tại nút (s),Trạng thái
STT,,,,,,,,,,
1,OD01,OD01,Miệng Hẻm 190 Nguyễn Gia Trí → cụm chân cầu Sà...,1.830,14.68,3,0.0,29,159.0,Có đường
2,OD02,OD02,Miệng Hẻm 336 Ung Văn Khiêm → cụm chân cầu Sài...,0.624,4.03,2,0.0,7,91.0,Có đường
3,OD03,OD03,Miệng Hẻm 97 Ung Văn Khiêm → miệng Hẻm 685 Điệ...,1.278,10.90,1,0.0,23,77.0,Có đường
4,OD04,OD04,Đài Liệt Sĩ → giao Ung Văn Khiêm – Nguyễn Gia Trí,0.488,4.40,0,0.0,12,30.0,Có đường
5,OD05,OD06,Miệng Hẻm 31 → miệng Hẻm 60 Ung Văn Khiêm,0.280,2.59,0,0.0,8,21.0,Có đường
6,OD06,OD07,Miệng Hẻm 42 Ung Văn Khiêm → miệng Hẻm 190 Ngu...,0.371,3.27,1,0.0,9,26.0,Có đường
7,OD07,OD13,Giao Ung Văn Khiêm – Nguyễn Văn Thương → cụm c...,0.573,5.08,0,0.0,11,30.0,Có đường
8,OD08,OD14,Miệng Hẻm 348 Ung Văn Khiêm → miệng Hẻm 685 Đi...,0.244,1.67,1,0.0,4,20.0,Có đường
9,OD09,OD17,Hàng Xanh → cụm chân cầu Sài Gòn,1.739,4.87,2,0.0,1,130.0,Có đường


## 8. Xuất mốc, tốc độ và kết quả

Ô cuối tạo ba tệp cấu hình và hai bảng CSV từ chính kết quả trong notebook. Bảng so sánh có 40 dòng cho 10 OD × (Dijkstra, Yen, BOA*, đường chính). Bảng chi tiết giữ thêm các tuyến Yen và tập Pareto BOA*. Cả hai bảng giữ mã OD mới và mã OD gốc. Nếu chọn chạy một phần OD, số dòng giảm tương ứng.


In [41]:
export_dir = Path("ket_qua_thu_nghiem_Thanh_My_Tay").resolve()
export_dir.mkdir(exist_ok=True)

def write_csv(name, records):
    with (export_dir / name).open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

write_csv("Moc_doan_ket_xe_Thanh_My_Tay.csv", data["markers"])
write_csv("so_sanh_ba_thuat_toan.csv", primary_results)
write_csv("chi_tiet_phuong_an.csv", alternatives)
for name, value in (("tuyen_duong_ket_xe.json", data["corridors"]),
                    ("toc_do_gia_dinh_chuan.json", data["profile"])):
    (export_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Đã xuất:", ", ".join(sorted(p.name for p in export_dir.iterdir())))

print("Thư mục xuất thực tế:", export_dir)
for filename, records in (("so_sanh_ba_thuat_toan.csv", primary_results),
                          ("chi_tiet_phuong_an.csv", alternatives)):
    with (export_dir / filename).open(encoding="utf-8-sig", newline="") as file:
        saved_rows = list(csv.DictReader(file))
    expected_rows = [{key: "" if value is None else str(value)
                      for key, value in row.items()} for row in records]
    if saved_rows != expected_rows:
        raise RuntimeError(f"Nội dung file không khớp kết quả trong bộ nhớ: {export_dir / filename}")
    od_ids = list(dict.fromkeys(row["Mã OD"] for row in saved_rows))
    print(f"{filename}: {len(saved_rows)} dòng | {len(od_ids)} OD | {', '.join(od_ids)}")


Đã xuất:

 Moc_doan_ket_xe_Thanh_My_Tay.csv, chi_tiet_phuong_an.csv, so_sanh_ba_thuat_toan.csv, toc_do_gia_dinh_chuan.json, tuyen_duong_ket_xe.json
Thư mục xuất thực tế: C:\Users\TP\Downloads\Bo_thu_nghiem_Thanh_My_Tay_10_OD_tu_ban_goc\Bo_thu_nghiem_Thanh_My_Tay\ket_qua_thu_nghiem_Thanh_My_Tay
so_sanh_ba_thuat_toan.csv: 40 dòng | 10 OD | OD01, OD02, OD03, OD04, OD05, OD06, OD07, OD08, OD09, OD10
chi_tiet_phuong_an.csv: 60 dòng | 10 OD | OD01, OD02, OD03, OD04, OD05, OD06, OD07, OD08, OD09, OD10
